# <center>OpenClaw 专题课第一节课：部署安装与核心架构</center>

&emsp;&emsp;很多人第一次接触 AI 助手时心里默认的画面是这样的："我发一句话，它处理，我等结果，处理完了我再发下一句。" 这套"一问一答、处理中不能打断"的认知，对绝大多数聊天机器人是对的。但今天我们要拆的 OpenClaw，恰恰在这一点上反着来：**它跑到一半，你能插话，而且你的话会插进当前这一轮，不是另起一个会话**。这一个差别，就是 agent 和 chatbot 的分水岭。

&emsp;&emsp;这一节课我们沿着同一条叙事主轴走完五章，主轴就是**一条真实消息的生命周期**——它从你嘴里发出，到 agent 把活干完为止，每一章都是这条主线上的一段剖面。第 0 章先认清"龙虾"到底是什么、不是什么，给后面四章打好语义地基；第 1 章带你在自己机器上把 OpenClaw 跑起来，发出第一条消息；第 2 章钻进 agent 运行时心脏，看清那个让它"能被打断"的双层 while 循环和它对外吐出的事件流；第 3 章看"心脏的手"——模型想调用一个工具时，这次调用要穿过几道关、凭什么有的放行有的拦截；第 4 章收口，把工具背后的安全档位讲全，并点透贯穿整套设计的张力哲学。用一句话串起来就是：**龙虾是什么（0）→ 在你机器上跑起来（1）→ 心脏怎么跳（2）→ 心脏的手怎么受控（3）→ 为什么这么设计（4）**，全程约 120 分钟。

&emsp;&emsp;贯穿这五章，我们会反复用两件武器：一边读真实的 TypeScript 源码锚点，一边用 Python 写最小可运行的重现版（MVP）在你眼前跑起来。读源码建立"它真的是这么写的"的信任，跑 MVP 建立"这个机制我能自己复现"的掌控感。其中第 2、3 章的机制最硬核，也是整节课的重心；第 0 章是认知热身，第 1 章是动手装机，第 4 章是哲学收束。下面我们从第 0 章开始，先把"龙虾"这个 OpenClaw 的吉祥物背后的身份认知建立起来。

> 📌 **目标受众与前置要求**：本课面向想真正搞懂 OpenClaw 是怎么跑起来的学员，从零开始带——第 0 章无任何前置；第 1 章需要你有基础命令行操作经验和 Node.js / pnpm 环境（课程会带你过一遍安装要求）；第 2、3、4 章前置你已经在本机跑通 OpenClaw（也就是第 1 章的产物），并且能读懂基础的 TypeScript 函数签名、能跑 Python 脚本。技术上你**不需要**事先理解 agent 内部循环，也**不需要**改动 OpenClaw 源码。如果你已有工程经验、读 TypeScript 没障碍，可以快速略过第 0、1 章直接进第 2 章的机制部分。

> 📌 **学完本节（约 120 分钟）你将带走 8 件产物**：① 能复述 OpenClaw 的跨渠道 × 跨设备 × 跨 model provider 三维定位，以及演进四代名称（Warelay → Clawdbot → Moltbot → OpenClaw）和每代的一句话定位变化；② 能在自己机器上从主分支源码跑通 `pnpm openclaw tui --local`，发出第一条消息并看到 TUI 把工具调用一个个亮出来；③ 一张能默画的 agent loop 双层 while + steering 插入示意；④ EventStream 全部 10 种事件（4 组生命周期）的名字，以及 6 种主骨架事件的先后顺序；⑤ 一句话说清 steering 为何是 agent 与 chatbot 的分水岭；⑥ plugin / capability / tool 三个概念的清晰边界——plugin 是带版本契约的宿主扩展单元、capability 是 plugin 的功能分类标签（15 种之一）、tool 是 plugin 注册暴露给 agent 调用的工具；⑦ 8 步工具策略管线的逻辑顺序，以及"exec 默认全开是有意设计而非漏洞"的正确框定；⑧ 能区分 exec `deny` / `allowlist` / `full` 三档安全级别的适用场景，以及 ask `off` / `on-miss` / `always` 三态的含义。

> 💡 **学完不能做（诚实划界）**：本节不会让你能从零实现一个生产级 agent runtime；课件里的 Python 代码是为了重现机制本质的最小模型，不等于 OpenClaw 的真实工程实现（真实实现有错误处理、流式、并发等大量细节我们不展开）。本节也不深入跨渠道 / 跨设备的 Gateway 骨架、记忆系统、多智能体协作——第 4 章只会点到这几条线的名字，不展开。

> 📅 **时效性说明**：本课全部源码引用基于 2026 年 5 月底 OpenClaw GitHub 主分支（地址 https://github.com/openclaw/openclaw）当时的代码状态。所有 `file:line` 引用都是真实可核对的——你可以在自己电脑 `git clone` 仓库后，用 `vim 路径 +行号` 打到对应位置亲自验证。

---

## <center>第 0 章：开场定位——OpenClaw龙虾是什么</center>

&emsp;&emsp;在动手装机和钻源码之前，我们先花十分钟把一件最容易被跳过、却最影响后续理解的事情做扎实——搞清楚 OpenClaw 到底是个什么东西。很多人看到它的吉祥物是一只龙虾，又听说它"能连微信能连 Telegram"，就先入为主地把它归类成"又一个聊天机器人"。这个第一印象一旦跑偏，后面讲 agent loop、讲工具系统时你会处处觉得别扭。所以这一章我们专门做认知校准：先给"龙虾"发一张身份证，说清它是什么（第 0.1 节）；再划清三条最常见的误解边界，说清它不是什么（第 0.2 节）；最后铺一张本节五章的路线图，让你知道接下来这趟旅程的地图长什么样（第 0.3 节）。

&emsp;&emsp;这一章没有任何代码，也不需要你的机器上装好任何东西——它是纯粹的认知地基。读完之后你应该能用三个维度复述清楚 OpenClaw 的定位，能说出它演进过程中换过的四个名字，也能在心里清楚地划掉三个常见的误会。我们从给龙虾发身份证开始。

### 0.1 龙虾的身份证：三维定位 + 四代演进

&emsp;&emsp;先看官方自己怎么定义。OpenClaw 的 `README.md` 开篇第一句话就是："OpenClaw is a _personal AI assistant_ you run on your own devices"——一个跑在你自己设备上的个人 AI 助手。注意这句话里藏着三个关键词：personal（个人的，不是多租户企业平台）、assistant（助手，能干活，不只是聊天）、your own devices（你自己的设备，在你的机器上跑）。把这三个词展开，就得到了理解 OpenClaw 最重要的**三维定位**。

&emsp;&emsp;第一维是**跨渠道**。OpenClaw 不绑定某一个聊天软件，它能接入二十多个你已经在用的消息平台——`README.md` 里列出的渠道包括 WhatsApp、Telegram、Slack、Discord、Signal、iMessage、Feishu（飞书）、WeChat（微信）、Microsoft Teams、Matrix 等等。也就是说，你在哪个软件里习惯收发消息，就能在那里跟它对话。第二维是**跨设备**。这里的"设备"指的是**你自己的多台电脑**（macOS / Linux / WSL 那种你能装软件的机器），而不是物联网传感器或嵌入式硬件——这一点很关键，我们下一节会专门拎出来纠正。第三维是**跨 model provider**。OpenClaw 不锁死在某一家大模型上，底层可以路由到不同的模型供应商。不过本节课我们用 `--local` 模式起一个最简的本地运行时，只涉及单个 provider，多 provider 路由不在本节展开。

&emsp;&emsp;认清了"它是什么"，我们再来看一个有意思的部分——它叫这个名字之前，还叫过别的。OpenClaw 这个项目经历了一段**四代演进**，这段历史不只是花絮，它恰好印证了"龙虾"这个隐喻的来历。我们直接看官方 `VISION.md` 第 13 行的原话："It evolved through several names and shells: Warelay -> Clawdbot -> Moltbot -> OpenClaw."（它经历了好几个名字和外壳：Warelay → Clawdbot → Moltbot → OpenClaw。）把这条演进线和官方 `docs/start/lore.md` 里记的故事对上，就能还原出每一代的来历。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102203159.png" width=50%></div>

&emsp;&emsp;这四代各有一句话能概括它当时的定位，我们一代一代看：第一代 **Warelay**——定位是"一个朴素的 WhatsApp 转发网关"，干的活就是把消息转来转去，名字平平无奇但够用。第二代 **Clawdbot**——定位是"龙虾 Clawd 入住后的 bot"，一只"太空龙虾"住了进来，项目以它命名（角色叫 Clawd）。第三代 **Moltbot**——定位是"蜕壳改名后的 Molty"，2026 年 1 月 Anthropic 发来一封礼貌的邮件，因为商标问题请求改名，于是龙虾做了龙虾最擅长的事：**蜕壳**（molt），蜕掉旧壳变成了住在 Moltbot 里的 Molty；可这个名字念起来总不够顺口。第四代 **OpenClaw**——定位是"2026 年 1 月 30 日最终定名，外壳变了、灵魂不变"，这个灵魂就是"能在真实电脑上干活的 agent 运行时"。一句话串起来：**网关（Warelay）→ 龙虾入住（Clawdbot）→ 蜕壳改名（Moltbot）→ 最终定名（OpenClaw）**，名字在变，那个"干活的 agent"内核始终没变。

&emsp;&emsp;这就引出了"龙虾"这个隐喻的精髓——**molting（蜕壳）**。龙虾通过一次次蜕壳来成长，每次换的是外壳，不变的是里面那个生命。OpenClaw 的演进正是如此：从 Warelay 到 OpenClaw，换了四个名字、四层外壳，但**里面那个"能在真实电脑上跑真实任务的 agent 运行时"的灵魂始终没变**。理解了这一点，你就抓住了 OpenClaw 的本质——它从第一天起，目标就不是"陪聊"，而是"干活"。

> **【常见误区】**：把 OpenClaw 理解成"又一个聊天机器人"。后果是你会用 chatbot 的框架去套它，从而完全无法理解后面第 2 章讲的 agent loop（一个能被中途打断的循环）和第 3 章讲的工具系统（能真实执行命令、读写文件）。正确理解是：OpenClaw 是一个 **agent 运行时 + 多渠道网关**，它的核心能力是跨渠道接入你的消息、然后在你的真实设备上执行真实任务。排查方法：每次你想说"它就是个 bot"时，问自己一句"一个普通的 bot 能帮我执行 shell 命令、读写我电脑上的文件吗"——答案是不能，而 OpenClaw 能。

### 0.2 不是什么：四条边界划清

&emsp;&emsp;认清了"是什么"，我们紧接着要做一件同样重要的事——划清"不是什么"。新接触一个项目时，最大的认知噪音往往不是"不知道它是什么"，而是"以为它是某个其实它不是的东西"。下面我们用一张对照表，把三条最高频的误解逐条纠正过来。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>OpenClaw 四条边界：常见误解 vs 正确理解</font></p>
<div class="center">

| 常见误解 | 实际情况（正确理解） | 依据 |
|----------|----------------------|------|
| OpenClaw 是一个"安装 AI 技能的应用商店"（Skill 平台）| skill 只是 OpenClaw 工具能力的一个维度（放在 workspace 下），不是 OpenClaw 本体的定位；它的本体是 agent 运行时+网关 | README 把 skills 列为 onboarding 的一项配套能力，与 channels / tools 并列 |
| OpenClaw 只是一个能连 Slack 的 bot | 它支持 WhatsApp / Telegram / Discord / Signal / iMessage / 飞书 / 微信等二十多个渠道，Slack 只是其中一个 | README 渠道清单列出 20+ 个 channel |
| 这里说的 device（设备）是 IoT 物联网传感器 | device 指**你自己装了 OpenClaw 的电脑**（macOS / Linux / WSL），不是物联网/嵌入式设备 | README:21 "run on your own devices" |
| ACP 是 OpenClaw 自己发明的协议 | ACP（Agent Client Protocol）是一个标准化「代码编辑器」与「编码 agent」通信的外部协议，OpenClaw 通过 `acpx` 插件做它的一个实现方，不是协议制定者 | npm 官方对 `@agentclientprotocol/sdk` 的描述 + OpenClaw 的 `acpx` 插件 |

</div>

&emsp;&emsp;这四条里，前三条是关于"定位范围"的纠正，第四条是关于"技术归属"的纠正，我们逐一说透。第一条，**OpenClaw 不是 Skill 平台**。你可能听过"给 AI 装技能（skill）"的说法，OpenClaw 确实支持 skill，但 skill 在它这里只是工具能力的一个组织方式，放在工作区里供 agent 调用——它绝不是 OpenClaw 的本体定位。把 OpenClaw 当成"技能商店"，就像把一台能装 App 的电脑当成"应用商店"一样，搞反了主次。

&emsp;&emsp;第二条，**OpenClaw 不是只能连 Slack 的 bot**。Slack 只是它支持的二十多个渠道之一。它真正的特点是"渠道无关"——你在哪儿习惯发消息，它就在哪儿待命。第三条，**device 不是 IoT 设备**。这个词在很多语境里指物联网传感器，但在 OpenClaw 里，device 特指你安装了它的那几台**能跑软件的电脑**。这一点直接关系到第 1 章——我们待会儿就是要在你的电脑（也就是一个 device）上把它装起来。

&emsp;&emsp;第四条值得多说一句，因为它涉及一个诚实划界。**ACP（Agent Client Protocol）不是 OpenClaw 自创的协议**。根据它的官方包描述，ACP 是一个用来标准化「代码编辑器」与「编码 agent」之间通信的协议（与 Zed 编辑器生态相关），OpenClaw 通过一个叫 `acpx` 的插件来对接它——也就是说，OpenClaw 是这个协议的**实现方之一**，而不是它的制定者。把这一点搞清楚，能帮你在后面遇到 `acpx` 这类名字时不至于误以为是 OpenClaw 的私有发明。

### 0.3 本节路线图

&emsp;&emsp;边界划清了，认知地基也就打好了。在正式进入装机环节之前，我们把这一节课的整张地图铺开看一遍，让你心里有数——接下来这一百二十分钟，我们会沿着哪条主线、分几站走完。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102203189.png" width=50%></div>

&emsp;&emsp;这张路线图的核心，是一条贯穿五章的主线——**一条真实消息的生命周期**。你可以把它想象成跟着一条消息走完全程：它从你嘴里发出（第 1 章你将亲手发出第一条），进入 agent 的心脏开始循环处理（第 2 章），循环中触发了工具调用、要穿过层层关卡（第 3 章），而这些关卡背后的安全档位和设计取舍（第 4 章）则解释了"为什么是这样设计的"。第 0 章则是这一切的起点——先认清这只龙虾的身份。

&emsp;&emsp;五站的节奏也值得提前交代：第 0 章（约 10 分钟）是认知热身，第 1 章（约 25 分钟）是动手装机，这两章是为后面铺路；第 2 章（约 40 分钟）和第 3 章（约 30 分钟）是整节课的硬核重心，我们会大量读源码、跑 MVP；第 4 章（约 15 分钟）是哲学收束，把工具背后的安全思想讲透并收口。

&emsp;&emsp;读完第 0 章，你应该已经能做到这几件事：用一句话说清龙虾是什么——一个跨渠道 × 跨设备 × 跨 model provider 的个人 AI agent 运行时（而不是聊天机器人）；说出它演进四代的名字和每代定位（网关 → 龙虾入住 → 蜕壳改名 → 最终定名）；以及干脆利落地划掉那四条边界——它不是 Skill 平台、不是只能连 Slack、device 不是 IoT、ACP 不是它自创的协议。这三样认知就是后面四章的地基。路线图心里有数了，我们就动手——下一章带你把 OpenClaw 在自己机器上跑起来。

---

## <center>第 1 章：本地安装部署——在你机器上跑起来</center>

&emsp;&emsp;上一章我们把"龙虾是什么"想清楚了，但光在脑子里想是不够的——一个 agent 运行时最有说服力的认知，永远来自"我亲手让它在我电脑上跑起来了"那一刻。这一章我们就干这件事：从零把 OpenClaw 的源码 clone 下来、装好依赖、配好模型、最后用一条命令把它跑起来，并发出你的第一条消息。当你看到它真的调用工具、把结果回填给你的时候，"它真的能在我机器上干活"这个信任感就建立起来了——而这正是后面第 2、3 章向内挖机制的动力来源。

&emsp;&emsp;这一章我们分四步走：先理解 OpenClaw 的代码是怎么组织的，也就是它为什么用 pnpm workspace 这种 monorepo 结构（第 1.1 节）；然后 clone 仓库并安装依赖（第 1.2 节）；接着配置模型 provider（第 1.3 节）；最后在仓库内用 `pnpm openclaw tui --local` 跑通并发出第一条消息（第 1.4 节）。需要先说明一件关于代码格式的事——这一章的代码 cell 跟第 2、3 章不一样。

> **【关于本章代码 cell 的特别说明】**：本章的命令演示是在一台已经装好 OpenClaw 的机器上真实运行的——它们调用的是真实的 `openclaw` 和 `pnpm` 命令。这些 cell 都用 Jupyter 的 `!` 前缀直接执行 shell 命令（`!` 是 Notebook 里"在终端跑一条命令"的标准写法，跟你在终端里手敲是一回事）。如果你的学习环境里还没装好 OpenClaw，直接运行会显示 `command not found: openclaw` 之类的提示——这完全正常，不影响你往下读，每个 cell 我们都在正文里标了"预期看到什么"。等你按本章步骤装好后，再回来真跑这些 cell 就能看到真实结果。

> 📅 **时效性说明**：本章涉及的安装命令和机制讲解，基于 2026 年 5 月底 OpenClaw GitHub 主分支当时的状态。OpenClaw 发布很勤，**你装到的版本号一定比录课时更新**——所以后文不再每次标注具体版本号，凡提到"版本"都指你当前装的那个，命令格式保持一致即可。

### 1.1 monorepo 技术栈：为什么是 pnpm workspace

&emsp;&emsp;在敲第一行命令之前，我们先花一点时间理解 OpenClaw 的代码是怎么摆放的——这会帮你理解后面为什么用 `pnpm` 而不是 `npm`。打开 OpenClaw 的 `README.md`，"From source"那一节开头有一句关键的话："The repository is a pnpm workspace"（这个仓库是一个 pnpm 工作区）。这句话告诉我们 OpenClaw 用的是 **monorepo（单仓多包）** 的组织方式。

&emsp;&emsp;什么是 monorepo？简单说，就是把多个本来可以拆成独立项目的代码包，放在同一个 Git 仓库里统一管理。OpenClaw 的仓库里既有 agent 运行时内核（我们第 2 章会重点读的 `agent-core`），也有各个渠道接入、各个插件扩展（`extensions/*` 下面那一堆）。用 `pnpm workspace` 把它们组织在一起，好处是依赖版本统一管理、子包之间可以直接互相引用、你改一处代码能立刻在整个工作区生效。这对一个"万物皆插件"的项目来说几乎是必然选择——它有大量需要解耦又需要协同的子模块。

&emsp;&emsp;这里有一个新手很容易踩的命令选择问题，我们提前讲清楚。

> **【常见误区】**：在源码 checkout 里直接用 `npm install`。后果是装不对——OpenClaw 的 `README.md` 明确写了"Plain `npm install` at the repo root is not a supported source setup"（在仓库根目录直接 `npm install` 不是受支持的源码安装方式），因为它的 bundled 插件是从 `extensions/*` 以 pnpm workspace 的方式加载的。正确做法：源码开发一律用 `pnpm`。排查方法：如果你 `npm install` 之后发现插件加载异常或依赖对不上，先检查自己是不是用错了包管理器。

&emsp;&emsp;还有一个值得记住的设计点，而且它直接决定了本章后面用哪条命令跑通——OpenClaw 其实有**两条运行路**。第一条是**全局发布版**：你可能之前通过包管理器装过一个全局的 `openclaw` 二进制（对应编译好的 `dist/` 产物），它跑的是某个**已发布的稳定版本**。第二条是**主分支源码版**：在你刚 clone 下来的仓库目录里用 `pnpm openclaw ...`，它通过 `tsx` 直接运行仓库里的 TypeScript 源码（改完即生效，跑的就是 GitHub 主分支的最新代码）；如果你需要一个编译好的 `dist/`（用于 Node 运行、打包或发布验证），则额外跑 `pnpm build`。

&emsp;&emsp;这两条路有一个**关键差别，本课的策略正是建立在它之上**：全局发布版是某个**已发布的稳定快照**，它的代码会**滞后于 GitHub 主分支**——而本课讲的很多机制、读的每一处源码行号，都是以主分支为准的。这意味着如果你用全局版跑本课的命令，可能遇到版本/行为对不上的情况：你读到的源码行号、某些默认值或日志格式，跟你装的发布版可能有出入。所以本章的策略是分工的：用全局 `openclaw --version` 查一下"系统里到底装没装过、装的是哪个发布版"（下面这格就干这件事）；但真正跑通发消息（1.4 节）会用**仓库内的 `pnpm openclaw`**（经 tsx 直接跑主分支源码），确保我们跑的代码跟课程讲解的源码**完全一致**。记住这个分工，它能帮你避开后面最容易翻车的一个坑。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603160435791.png" width=70%></div>

&emsp;&emsp;铺垫到这里，我们用一条最简单的命令查一下系统里**全局发布版** OpenClaw 的情况——查它的版本号。注意这一步查的是全局 `openclaw` 二进制（也就是上面说的第一条路），目的是看清"系统里装没装过、装的是哪个发布版"，它和我们 1.4 节真正用来跑通的 `pnpm openclaw` 不是同一条路。下面这格用 Jupyter 的 `!` 前缀直接执行 `openclaw --version`，运行后你会看到一行 OpenClaw 的版本号输出；如果显示 `command not found`，说明你的系统里还没装过全局发布版——这完全没关系，因为本课跑通靠的是仓库内的源码版，跟着后面 1.2 节 clone 源码即可。

In [3]:
# 本 cell 在本机真实运行；学习环境未装 openclaw 时会显示 command not found，属正常现象
# 查 OpenClaw 全局版本，确认系统里装没装过、装的是哪个发布版
!openclaw --version
# 预期看到这样一行：OpenClaw 后面跟你当前安装的版本号

OpenClaw 2026.5.28 (e932160)


&emsp;&emsp;这格做的事很简单——用 `!` 前缀直接在终端跑一条 `openclaw --version`。`!` 是 Jupyter 的约定：以它开头的行会被当作 shell 命令交给系统执行，跟你在终端里手敲完全一样，本章后面所有命令演示都用这种写法。如果你还没装 OpenClaw，这格会显示 `command not found`，不用慌，这正说明"系统里还没有全局发布版"；在一台装好的机器上真跑时，会输出当前安装的 OpenClaw 版本号。下一节我们就开始真正的安装。

### 1.2 clone 仓库 + 安装依赖

&emsp;&emsp;现在进入实际安装。这一步的目标是把 OpenClaw 的源码弄到你机器上，并装好它运行需要的所有依赖。在动手之前，有一个前置条件必须先确认——Node.js 的版本。

> **【踩坑预警】**：Node.js 版本不达标会导致后续 `pnpm install` 或运行时报错。OpenClaw 的 `README.md` 明确写了运行时要求："Node 24 (recommended) or Node 22.19+"（推荐 Node 24，或 Node 22.19 以上），它的 `package.json` 里 engines 字段也写死了 `node >=22.19.0`。后果是：如果你的 Node 版本太旧，依赖安装阶段就会失败，或者装上了但跑不起来。正确做法：动手前先 `node --version` 检查，低于 22.19 就先升级 Node——推荐用 `nvm`（Node 版本管理器，`nvm install 24` 一条命令装好并切换），或直接去 nodejs.org 下载 Node 24 安装包。排查方法：遇到安装或启动报错，第一件事就是回头核对 Node 版本。

### 1.2.1 按系统选择安装路径

&emsp;&emsp;正式 clone 源码之前，我们先把"安装 OpenClaw"这件事分清楚。OpenClaw 官方其实给了两类路径：一类是**普通用户安装**，目标是尽快把全局 `openclaw` CLI 和 Gateway daemon 装好；另一类是**源码开发 / 课程学习安装**，目标是从 GitHub checkout 里直接跑主分支 TypeScript 源码。本课后面要读源码、跑 `pnpm openclaw tui --local`，所以我们采用第二类路径；但如果你只是想先把 OpenClaw 当工具用，第一类路径会更省事。

<p align="center"><font face="黑体" size=4>OpenClaw 不同系统的推荐安装路径</font></p>

| 系统环境 | 官方可用路径 | 本课推荐做法 | 关键提醒 |
| --- | --- | --- | --- |
| macOS / Linux | 官方安装脚本：`curl -fsSL https://openclaw.ai/install.sh \| bash`；也可以用 `npm install -g openclaw@latest` 或 `pnpm add -g openclaw@latest` | 如果跟本课学源码机制，走下面的 `git clone` + `pnpm install` + `pnpm openclaw setup` | 普通安装不一定需要你手动管理 pnpm；源码 checkout 才必须用 pnpm |
| Windows 原生 PowerShell | 官方安装脚本：`iwr -useb https://openclaw.ai/install.ps1 \| iex` | 只想体验 CLI / Gateway 时可以走原生安装；要做源码学习不作为首选 | 官方说明原生 Windows 可用，但仍有 caveats，完整体验优先考虑 WSL2 |
| Windows + WSL2 | 在 WSL2 里按 Linux 流程安装；官方也明确把 WSL2 标为更稳定路径 | 本课最推荐的 Windows 路线：先装 WSL2 + Ubuntu，再在 WSL 里执行后续源码步骤 | 后续命令里的 `git clone`、`pnpm install`、`pnpm openclaw ...` 都应在 WSL 终端内运行 |

&emsp;&emsp;也就是说，如果你是 macOS 或 Linux 学员，可以直接继续往下走；如果你是 Windows 学员，建议先判断自己是哪种目标：只是普通使用，可以在 PowerShell 里跑官方 `install.ps1`；要跟着本课读源码、跑主分支新功能，建议先进入 WSL2 的 Ubuntu 环境，再按本章后面的 Linux 源码流程执行。这样做的原因很简单：OpenClaw 当前同时支持 native Windows 和 WSL2，但官方文档明确说 **WSL2 是更稳定、体验更完整的路径**。

In [2]:
#!curl -fsSL https://openclaw.ai/install.sh

> **【踩坑预警】**：把 Windows PowerShell 和 WSL2 终端混在一起用。后果是路径、Node、pnpm、全局命令位置全都可能对不上——你在 PowerShell 里装的 `openclaw`，不等于 WSL 里也有；你在 WSL 里 clone 的仓库，PowerShell 也不一定在同一个工作目录下。正确做法：选定一条路线后保持一致。本课源码路线建议 Windows 学员全程在 WSL2 里操作；只有明确走原生安装时，才在 PowerShell 里执行 `install.ps1`。

### 1.2.2 装好包管理器 pnpm

&emsp;&emsp;路径选好了，正式 `clone` 之前还有最后一块环境拼图要补上——`pnpm` 本身。上一节的表格里我们已经点了一句关键提示：普通安装不一定需要你手动管理 `pnpm`，但**源码 checkout 必须用 `pnpm`**。既然本课走的是源码路线，那么在敲 `git clone` 之前，就得先确认你的机器上有 `pnpm` 这条命令，否则到了下面"步骤二"的 `pnpm install` 那一格，你会直接撞上 `pnpm: command not found`。

&emsp;&emsp;这里要先厘清一个新手很容易混的点：`Node.js`、`npm`、`pnpm` 三者并不是平级的"三样都得单独装"。`npm` 是 `Node.js` **自带**的包管理器——你在上一步装好 `Node.js` 时，`npm` 就已经一起到位了，不需要单独安装；而 `pnpm` **不随 `Node.js` 自带**，它是一个独立的第三方包管理器，必须你手动装这一步。换句话说，本节真正要"动手装"的只有 `pnpm` 一个。

&emsp;&emsp;`pnpm` 官方给了三条安装路径，任选其一即可。下面这张表把它们的命令、适用场景和注意点放在一起，方便你对照自己的系统挑一条最省事的：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>pnpm 三种官方安装方式对比</font></p>
<div class="center">

| 安装方式 | 命令 | 适用场景 | 关键提醒 |
| --- | --- | --- | --- |
| Corepack（推荐） | `corepack enable pnpm` | 已按上一步装好 `Node.js`，想要最省心 | `Node.js` 自带 Corepack，无需额外下载；首次启用若报签名过期，先升级 Corepack 再试 |
| npm 全局安装 | `npm install -g pnpm` | 习惯用 `npm`、不想碰 Corepack | 用的就是上一步 Node 自带的 `npm`；若报 `EACCES` 权限错误，优先改用 Corepack 或换 `nvm` 安装的 Node |
| 独立脚本 | `curl -fsSL https://get.pnpm.io/install.sh \| sh -` | Linux / Apple Silicon macOS，想要不依赖 Node 的独立安装 | **Intel 芯片的 macOS 不适用**，请改用 Homebrew 或上面两种方式 |

</div>

&emsp;&emsp;如果你拿不准选哪条，直接用第一条 `corepack enable pnpm`——Corepack 是 `Node.js` 官方内置的包管理器版本管理工具，你装完 Node 它就已经在了，一条命令就能把 `pnpm` 激活，是三种里最不容易出岔子的。这里要补一个版本边界：Corepack 从 `Node.js` 25 起不再随 Node 一起分发，本课推荐的 Node 24（以及 22）仍然自带它；如果你用的恰好是 Node 25 及以上，改用上面的 `npm install -g pnpm`、或按官方文档单独安装 Corepack 即可。

In [ ]:
# 本 cell 在本机真实运行；学习环境若尚未装 pnpm 会显示 command not found，属正常现象，按上面任一方式装好后再跑即可
# 校验 pnpm 是否就位并查看版本号
!pnpm --version
# 预期看到：一个版本号（如 9.x / 10.x）；若提示 command not found，说明 pnpm 还没装上

&emsp;&emsp;这一格做的事很简单——用 `pnpm --version` 确认 `pnpm` 这条命令真的能被终端找到。只要它打印出一个版本号，就说明环境前置已经齐了：`Node.js` 在前一步备好、`pnpm` 在这一步装上，接下来就可以放心进入"步骤一：克隆仓库"。如果这里没打印版本号、反而报 `command not found`，分两种情况处理：要么是还没装，回到上面的表格任选一条命令装好；要么是你确认装过了却仍然找不到，这多半是当前终端或 Jupyter 还在用旧的 `PATH`——重开一个终端、或重启 Notebook 的 kernel 之后，再跑这一格即可。

> 🔥 **踩坑预警 · Corepack 首次启用报签名过期**：用 `corepack enable pnpm` 时，部分机器会报一个 signature / 签名校验相关的错误。后果是命令中断、`pnpm` 没装上。原因是随 `Node.js` 分发的 Corepack 版本偏旧。正确做法：先 `npm install -g corepack@latest` 把 Corepack 升到最新，再重新执行 `corepack enable pnpm`。排查方法：如果 `corepack` 报错、而直接 `npm install -g pnpm` 能正常装上，基本就是这个问题，换用 npm 方式或升级 Corepack 即可。

**步骤一：克隆仓库**

&emsp;&emsp;第一步是把 OpenClaw 的代码从 GitHub 克隆到本地。这一步对应 `README.md`"From source"小节里的第一条命令。下面这段代码演示克隆过程，运行后你会看到 git 把仓库拉取下来的进度输出；如果目标目录已存在，git 会提示已存在而不会重复克隆。

In [ ]:
# 本 cell 在本机真实运行，学习环境因 git/网络差异结果不同
# 克隆 OpenClaw 源码仓库（命令来源：README.md From source 小节）
!git clone https://github.com/openclaw/openclaw.git
# 预期看到：Cloning into 'openclaw'... 及拉取进度；若目录已存在会提示 already exists

&emsp;&emsp;这条命令把整个 OpenClaw 仓库拉到当前目录下的 `openclaw/` 文件夹里。克隆完成后，你的机器上就有了我们后面要读的所有源码——包括第 2 章的 `agent-loop.ts`、第 3 章的 `tool-policy-pipeline.ts` 这些文件。接下来进入仓库目录安装依赖。

**步骤二：安装工作区依赖**

&emsp;&emsp;源码拿到手后，要装好它运行所需的依赖包。因为前面讲过 OpenClaw 是 pnpm workspace，所以这里**必须用 `pnpm install` 而不是 `npm install`**。这一步会把整个工作区（包括各个子包和 bundled 插件）的依赖一次性装好。下面这段代码在 `openclaw` 目录里执行安装，运行后你会看到 pnpm 解析依赖、下载、链接的进度。

In [ ]:
# 本 cell 在本机真实运行，学习环境因 openclaw 源码未 clone 跑不通
# 在 openclaw 仓库目录里安装所有工作区依赖（必须用 pnpm，不能用 npm）
# 注意：! 命令里的 cd 只在本行子进程生效，所以用 cd openclaw && ... 连写
!cd openclaw && pnpm install
# 预期看到：pnpm 解析依赖 + 下载 + 链接的进度，结尾 Done

&emsp;&emsp;这条命令完成后，OpenClaw 的依赖就齐了。注意我们用 `cd openclaw &&` 先切进仓库目录再执行 `pnpm install`——如果你在别的目录跑，pnpm 找不到 workspace 配置就会出错。（在 Notebook 里 `!cd` 不会改变后续 cell 的工作目录，所以每条需要在仓库内执行的命令都用 `cd openclaw && ...` 连写。）

**步骤三：首次设置与运行准备**

&emsp;&emsp;依赖装好后，源码方式运行还需要做一次首次设置，写好本地配置和工作区。`README.md` 给出的开发循环里，首次运行要先执行 `pnpm openclaw setup`（它会写好 `pnpm gateway:watch` 所需的本地配置/工作区，可以安全地重复执行）；如果你需要一个编译好的 `dist/` 产物，则额外跑 `pnpm build`。下面这段代码演示首次设置。

In [ ]:
# 本 cell 在本机真实运行，学习环境因依赖未装跑不通
# 首次设置：写好本地 config/workspace（命令来源：README.md From source 小节，可安全重复执行）
!cd openclaw && pnpm openclaw setup
# 预期看到：写入本地 config/workspace 的确认信息

&emsp;&emsp;这一步把本地配置和工作区准备好——之后 OpenClaw 才知道去哪儿读你的设置、把会话状态存在哪儿。`pnpm openclaw setup` 是幂等的，重复跑不会出问题，通常你只在第一次安装或重置本地状态后需要它。到这里，OpenClaw 在你机器上已经"装好了"，下一步是告诉它用哪个大模型。

### 1.3 配置模型 provider

&emsp;&emsp;OpenClaw 是一个 agent 运行时，它的"大脑"是大模型——所以在跑起来之前，你得先告诉它用哪家的模型、用什么凭证去访问。这一步就是配置 model provider。OpenClaw 提供了一个交互式的配置命令 `openclaw configure` 来引导你完成这件事。

&emsp;&emsp;在演示配置命令之前，有一条安全纪律必须强调——**API key 绝对不要写进代码 cell 或提交到 Git**。OpenClaw 的 `configure` 命令会引导你交互式地输入凭证并安全地存到本地配置里，你不需要、也不应该把 key 明文写在任何课件代码里。下面这段代码查看 `configure` 命令的帮助，让你了解它能配哪些东西，运行后你会看到 configure 子命令的用法说明。

In [4]:
# 本 cell 在本机真实运行，学习环境因 openclaw 未装跑不通
# 查看 configure 命令帮助，了解可配置项（provider / key 等通过交互式向导设置）
!openclaw configure --help
# 预期看到：configure 子命令的用法与可配置项说明


🦞 OpenClaw 2026.5.28 (e932160) — All your chats, one OpenClaw.

Usage: openclaw configure [options]

Interactive configuration for credentials, channels, gateway, and agent defaults

Options:
  -h, --help           Display help for command
  --section <section>  Configuration sections (repeatable). Options: workspace,
                       model, web, gateway, daemon, channels, plugins, skills,
                       health (default: [])

Docs: ]8;;https://docs.openclaw.ai/cli/configuredocs.openclaw.ai/cli/configure]8;;



<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603135344431.png" width=50%></div>

&emsp;&emsp;这条命令展示的是配置入口的帮助信息。真正配置时，你会按向导一步步选 provider、填 key——这些都在交互过程里完成，不进代码。课堂演示用的是本机已经配好的 provider；你按照自己手上有 key 的那家（比如某个你能访问的模型供应商）配即可。

&emsp;&emsp;这里正好能看到第 3 章要讲的一个机制的真实预演。举个例子：当你装的某个插件要求的宿主版本高于你当前的全局版本时，跑 OpenClaw 命令会打印出这样一行日志：`plugin requires OpenClaw >=2026.4.25, but this host is 2026.3.24; skipping load`（这个插件要求 OpenClaw 2026.4.25 以上，但当前主机更低，于是跳过加载）。这正是第 3 章要讲的 **`minHostVersion` 版本契约门控**的真实表现：每个插件声明自己最低需要哪个版本的宿主，宿主太旧时插件被安全地"跳过加载"而不是让整个程序崩溃。你现在不用深究，记住这个现象，第 3 章我们会回头解释它的机制。

&emsp;&emsp;`configure` 让我们看清了"怎么配"，但配置这件事还有同样重要的另一半——**怎么确认我们配的到底有没有生效、又是从哪个文件生效的**。这一点在 OpenClaw 上尤其值得讲清楚，因为它的 provider 配置不是只存在一个文件里，而是**分两层存放**的；如果你之前借助某些第三方工具管理过模型 key，很可能遇到过"我明明改了 key，怎么没生效"的困惑。OpenClaw 给了我们一个专门的体检命令 `openclaw models status`，它会把每个 provider **实际生效的凭证来源**一行行列出来。我们先把它跑起来看一眼，再对着输出把两层配置的关系讲透。

In [5]:
# 本 cell 在本机真实运行，学习环境因 openclaw 未装/未配 provider 跑不通
# 给 provider 配置做一次"体检"：看默认模型、兜底链，以及每个 provider 实际生效的凭证来自哪个文件
!openclaw models status
# 预期看到：Default 默认模型、Fallbacks 兜底链，以及每个 provider 的 effective= 与 source= 行


🦞 OpenClaw 2026.5.28 (e932160)
   I read logs so you can keep pretending you don't have to.

Config        : ~/.openclaw/openclaw.json
Agent dir     : ~/.openclaw/agents/main/agent
Default       : minmax/MiniMax-M3
Fallbacks (1) : kimicoding/kimi-for-coding
Image model   : -
Image fallbacks (0): -
Aliases (4)   : Opus -> tokenmax/claude-opus-4-6, sonnet -> anthropic/claude-sonnet-4-6, Nova -> yunyi/claude-opus-4-6, Kimi -> kimicoding/kimi-for-coding
Configured models (4): tokenmax/claude-opus-4-6, anthropic/claude-sonnet-4-6, yunyi/claude-opus-4-6, kimicoding/kimi-for-coding

Auth overview
Auth store    : ~/.openclaw/agents/main/agent/auth-profiles.json
Shell env     : off
Providers w/ OAuth/tokens (1): openai-codex (1)
- anthropic effective=profiles:~/.openclaw/agents/main/agent/auth-profiles.json | profiles=1 (oauth=0, token=0, api_key=1) | anthropic:default=sk-e572c...21af5a93
- kimicoding effective=models.json:sk-kimi-...0qOFLjKn | models.json=sk-kimi-...0qOFLjKn | source=models.j

&emsp;&emsp;先看这条命令的输出。最上面几行是全局视角：`Default` 是当前默认模型、`Fallbacks` 是兜底链。真正的重点在下面——每个 provider 各占一行，尤其是每行末尾的 `effective=` 和 `source=` 两个字段。`effective=` 是这个 provider **当前实际在用的凭证**，`source=` 则告诉你这份凭证**是从哪个文件读出来的**。换句话说，`models status` 不是挑某一个配置文件读给你看，而是把 OpenClaw 真正"认"的那份配置摊开给你——这正是它比我们自己去 `cat` 某个 `json` 文件更可靠的地方。

&emsp;&emsp;为什么要专门强调"从哪个文件读出来的"？因为 OpenClaw 的 provider 配置是**分两层**存放的，搞不清这两层的关系，是新手在配置阶段最容易栽的一个坑。我们先用一张表把这两层摆清楚。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>OpenClaw provider 配置的两层结构</font></p>
<div class="center">

| 配置层 | 文件路径 | 作用范围 | 优先级 |
|--------|----------|----------|--------|
| 全局层 | `~/.openclaw/openclaw.json` | 所有 agent 共享的默认 provider | 低 |
| agent 级 | `~/.openclaw/agents/<agentId>/agent/models.json` | 单个 agent 私有的 provider | 高（覆盖全局层） |

</div>

&emsp;&emsp;这两层的关系一句话就能概括：**agent 级覆盖全局层**。OpenClaw 的配置里有一个 `models.mode: "merge"` 的设定，意思是它会把全局层和 agent 级两份 provider **合并**起来用；一旦同一个 provider 在两层都出现，**agent 级的值会盖过全局层**。设计成这样是有道理的——OpenClaw 支持同时跑多个 agent，每个 agent 可能想用不同的模型供应商、不同的 key，所以每个 agent 都能在自己的 `models.json` 里覆盖全局默认。这也是为什么你在 `models status` 里看到的 `source=` 大多指向 agent 级的 `models.json`，而不是那个全局的 `openclaw.json`。理解了这一点，那个"改了 key 却不生效"的困惑就有了答案。

> 🔥 **踩坑预警 · 改了配置却不生效**：如果你借助第三方的 key 管理 / provider 切换工具（比如 `cc-switch` 这类）来配 OpenClaw 的模型，要特别留意它把配置写进了**哪一层**。这类工具往往只负责更新其中**一层**（常见是全局层 `openclaw.json`），可你的 agent 实际读的是 **agent 级** `models.json`——两层不一致时 agent 级优先，于是工具里改的新 key 根本没被用上。后果是你以为换了 key、其实还在用旧的，排查半天找不到原因。正确做法：改完任何 provider 配置，都用 `openclaw models status` 看一眼对应 provider 的 `source=` 和 `effective=`，确认实际生效的文件和凭证就是你刚改的那一份。排查方法：如果 `source=` 指向的文件跟你刚改的那个不是同一个，说明你改错了层，到 `source=` 指的那个文件里改才有效。

&emsp;&emsp;所以这一节我们带走的不只是"用 `configure` 配 provider"这一个动作，还有一套**确认配置真正生效**的习惯：配完用 `models status` 体检，认准 `source=` 那一层。把这套习惯养成，后面无论是自己手改 `json`、还是借助工具管理 key，你都能一眼看清 OpenClaw 到底在用哪份配置——这正是"配得对"和"配得明白"的区别。

### 1.4 跑通 `pnpm openclaw tui --local`：发出第一条消息

&emsp;&emsp;万事俱备，到了最激动人心的一步——把 OpenClaw 真正跑起来，发出你的第一条消息。这里要特别提醒：跑通这一步，我们用的是**仓库内的 `pnpm openclaw tui --local`**，而不是全局的 `openclaw`。原因前面 1.1 节说过——我们要跑的是跟课程讲解**完全一致**的主分支源码，而全局发布版是已发布的稳定快照、会滞后于主分支，行为可能对不上。所以请确保你是**在 1.2 节 clone 下来的那个 `openclaw` 仓库目录里**，用 `pnpm openclaw`（经 tsx 直接跑主分支源码）来执行，这样跑的就是带 `--local` 的当前主分支代码。这条命令的两个部分都值得说一说。

&emsp;&emsp;先看 `tui`。它是 OpenClaw 的终端界面（Terminal UI）子命令，在源码 `src/cli/tui-cli.ts` 第 10 行定义。有意思的是它还有两个别名——`terminal` 和 `chat`，所以 `pnpm openclaw chat --local` 和 `pnpm openclaw terminal --local` 跟 `pnpm openclaw tui --local` 是完全等价的，你用哪个都行。再看 `--local`。这个标志的含义是"针对本地内嵌的 agent 运行时运行"（源码里的原文描述是 "Run against the local embedded agent runtime"）——也就是说，它**绕过 Gateway（网关）**，直接起一个最简的本地 agent 运行时。这对我们学习再合适不过：不用去配什么网关、什么远程连接，一条命令就能在本地把 agent 跑起来。

&emsp;&emsp;这里还藏着一层和本章策略呼应的好处。加上 `--local`，真正执行 agent 的就是你当前用 `pnpm openclaw` 起的这个**源码版进程自己**，跑的正是课程逐行讲解的主分支代码。反过来，如果不加 `--local`，`tui` 只会去连一个**已经在后台运行的 Gateway 守护进程**，真正干活的是那个进程——而它未必是你 clone 的源码，很可能是你早先用全局发布版起的常驻服务。所以 `--local` 对我们不只是"图省事、少配网关"，它还顺手锁定了一件要紧的事：**你执行的代码，就是你正在读的这份源码**。这正是 1.1 节那个"用源码版保证跟讲解完全一致"策略的延伸——不光命令前缀要用 `pnpm openclaw`，真正跑起来的运行时也得是它。

&emsp;&emsp;我们已经反复提到 Gateway，这里花一小段把它说清楚——毕竟 `--local` 的全部意义就是"绕过它"。**Gateway 是 OpenClaw 那个一直在后台运行的常驻进程**（一个 WebSocket 服务），也是第 0 章说的"跨渠道 × 跨设备"真正落地的中枢：你在 WhatsApp、Telegram、飞书等渠道发来的消息都汇聚到它，由它来跑 agent、调度定时任务，再把结果发回对应渠道——它就是那句口号"All your chats, one OpenClaw（所有聊天，汇于一个 OpenClaw）"的技术承载者。正因为要随时接消息、被各台设备连接，它被设计成常驻在线的守护进程。而本课用 `--local` 绕过它，是因为我们这一节只想"在本机把 agent loop 跑起来看明白"，用不上多渠道接入、定时任务、远程连接这些中枢能力，内嵌运行时已经够了。至于 Gateway 完整的职责全貌——它怎么在多个渠道、多台设备之间路由分配，怎么守住安全边界——超出本节范围，这里你只要记住"Gateway = 常驻中枢，`--local` = 绕过它、单机直跑"这一组对照就够了。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102319175.png" width=50%></div>

&emsp;&emsp;还有一个调试技巧值得记住：如果你想看到更详细的内部日志，可以加 `--log-level debug`。但要注意它的位置——`--log-level` 是一个根级选项，必须放在子命令 `tui` **前面**。

> **【踩坑预警】**：把 `--log-level debug` 放在 `tui` 后面。后果是选项不生效，你看不到想要的调试日志。正确写法是 `pnpm openclaw --log-level debug tui --local`——`--log-level` 在前，子命令在后。顺序反了（写成 `pnpm openclaw tui --local --log-level debug`）不会按预期工作。排查方法：记住"根级选项在子命令前"这条规则；另外 `debug` 是 OpenClaw 合法的日志级别之一（源码 `src/logging/levels.ts` 里 `ALLOWED_LOG_LEVELS` 包含 `debug`），拼错级别名也会失效。

> **【踩坑预警 · 最容易翻车的一个】**：用全局的 `openclaw` 跑本课命令（而不是仓库内的 `pnpm openclaw`）。后果是你跑的是某个已发布的稳定快照，它的代码滞后于主分支——你读到的源码行号、某些默认值或日志格式，可能跟课程讲解对不上。这不是你配置错了，而是版本差异。正确做法：在 1.2 节 clone 的 `openclaw` 仓库目录里，用 `pnpm openclaw tui --local`（经 tsx 直接跑主分支源码，跟课程讲解的源码完全一致）。排查方法：遇到行为对不上时，先确认两件事——一是你在不在仓库目录里，二是命令前缀是不是 `pnpm openclaw` 而非裸 `openclaw`。

&emsp;&emsp;现在我们发出第一条消息。`tui` 默认是交互式的（起一个界面等你输入），但为了在 Notebook 里一格跑完就能看到输出，我们用 `--message` 标志单次发送一条消息。我们精心挑了一条任务消息——"查一下现在几点，把结果写进 a.txt"。为什么是这一条？因为它需要 agent 真的去**调用工具**完成：取当前时间、再把结果落进文件。**具体调几个工具、怎么调，是 agent 自己拆解决定的**——有的模型用一条 `exec`（类似 `date > a.txt` 的 shell 重定向）一步搞定，有的会分成"取时间"和"写文件"两步。不管哪种，你都会看到工具被一个个亮出来、结果回填，这就是一次完整的工具调用 loop，而这正是第 2 章要向内剖析的东西。注意下面这格用 `cd openclaw &&` 切进仓库目录、以 `pnpm openclaw` 的方式执行——这是确保跑的是带 `--local` 的主分支源码的关键。运行后你会在 TUI 的输出里看到：模型先想，然后调用工具（工具名 + 参数会被亮出来），工具执行完结果回填，最后给你一个答复。

In [ ]:
# 本 cell 在本机真实运行，学习环境因 openclaw 源码未 clone/未配 provider 跑不通
# 关键：在 clone 的 openclaw 仓库目录内用 pnpm openclaw（经 tsx 跑主分支源码，跟课程讲解一致）
# 而非全局 openclaw（发布版滞后于主分支，行为可能对不上）
!cd openclaw && pnpm openclaw tui --local --message "查一下现在几点，把结果写进 a.txt"
# 预期看到：模型回复 + agent 调用工具完成任务（可能一条 exec 直接写文件，也可能 exec/write 两步）的展示

&emsp;&emsp;这条命令是本章的高潮——你的第一条消息真正驱动了一次完整的 agent 工作。运行后在 TUI 里，你会看到 OpenClaw 不是简单地回你一句话，而是**真的去执行了任务**：调用工具取系统时间、把结果写进 `a.txt`（具体用一条 `exec` 重定向、还是 `exec`/`write` 两步，由模型当场决定）。如果你跑通了这一步，看到工具被一个个亮出来、结果被回填，那么"OpenClaw 真的能在我机器上干活"这个信任感就建立起来了。

> 🔥 **踩坑预警 · tui 是交互式界面，在 Notebook 里会挂起**：`tui` 是一个交互式全屏界面，不像前面几条命令跑完就结束——即使我们用 `--message` 发完了初始消息，它也不会自动退出。所以你在 Jupyter 里直接跑这格，可能渲染不全、甚至一直挂着不返回（cell 左侧一直显示 `[*]`）。想完整体验 TUI，建议把这条命令复制到真实终端里跑；如果在 Notebook 里跑卡住了，点工具栏的中断按钮（■）停掉即可，不影响你往下学。

> **【预期输出示意 · 怎么判断跑对了】**：零基础的你可能不确定"什么样算成功"，给你一个对照——跑对时，TUI 里大致会依次出现这几样：① 模型先输出一段思考/计划文字；② 一个工具名被亮出来（比如 `exec`）连同它的参数；③ 这个工具的执行结果回填显示出来；④ 接着可能再亮一个工具（`write`）并写出 `a.txt`；⑤ 最后模型给你一句完整答复（类似"已查到当前时间并写入 a.txt"）。如果你只看到一句纯文字回复、完全没有工具被亮出来，那多半是消息没触发工具（换成我们这条"查时间+写文件"的任务消息再试）；如果直接报错退出，回头检查 1.3 的 provider 是否配好、命令是不是在仓库目录里用 `pnpm openclaw` 跑的。

&emsp;&emsp;跑通这一步，意味着你已经把这只"龙虾"在自己机器上真正养活了——从一行 clone 命令开始，到亲手发出第一条消息、看着它调用工具把活干完，这是整节课最有里程碑感的一步。后面第 2、3 章所有向内挖机制的内容，都站在你此刻这个"它真的能跑"的地基上。现在我们做一次对账，把你刚在 TUI 上看到的现象，跟第 2 章即将剖析的内部机制对应起来——这也是本章到下一章的桥梁。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>你在 TUI 上看到的 × 第 2 章将剖析的内部机制</font></p>
<div class="center">

| 你在 TUI 上看到的现象 | 它对应第 2 章的什么机制 |
|------------------------|--------------------------|
| agent 开始处理你的消息 | `agent_start` 事件，loop 启动 |
| 模型想了一下，决定调用 `exec` | 模型输出 + 进入工具调用，`turn_start` |
| TUI 亮出 `exec` 工具名和参数 | `tool_execution_start` 事件被 TUI 订阅渲染 |
| `exec` 跑完，结果回填 | 工具结果挂在 `turn_end` 的 `toolResults` 字段 |
| 接着又调用 `write` 写文件 | 同一次 loop 内的下一轮工具调用 |
| 最后给你一个完整答复 | `agent_end` 事件，本次处理收尾 |

</div>

&emsp;&emsp;这张表就是本章和第 2 章之间的"伏笔"。你现在看到的是**现象**——工具一个个亮起、结果一个个回填；而第 2 章我们要回答的是**机制**——这些现象背后，是一个怎样的循环在驱动、一条怎样的事件流在 emit。带着你刚才在 TUI 上的亲眼所见，我们正好可以追问一句：消息进来之后，这个 loop 到底是怎么转的？这正是下一章要钻进去看的。不过在钻进机制之前，本章还剩两件"打基础"的事要补上：一是我们刚才跑通的 `tui --local` 只是 OpenClaw 的一条路（在终端里、绕过 Gateway），它其实还备了另一条更直观的路——浏览器图形界面，下一节就带你把它打开（第 1.5 节）；二是把日常最常用的几招玩法过一遍（第 1.6 节）。两件事都铺平了，我们再安心钻进 agent 的心脏。

### 1.5 另一条路：浏览器 Control UI

&emsp;&emsp;上一节我们用 `pnpm openclaw tui --local` 在终端里跟 agent 对上了话，但你可能会想——一个号称"跨设备"的 agent 运行时，难道只能在黑乎乎的终端里敲命令吗？并不是。OpenClaw 还提供了一个**浏览器里的图形控制台**，官方叫它 `Control UI`，它有一个真正的对话框，能像聊天软件一样跟 agent 收发消息，还能在侧边栏查看会话、活动、定时任务等一整套面板。这一节我们就把它打开，并顺手补全上一节那张"Gateway 拓扑图"里**没走到的那一半路**。

&emsp;&emsp;先说清楚 Control UI 到底是什么、它跟上一节的 tui 是什么关系。Control UI 是一个用 `Vite` + `Lit` 写的浏览器单页应用，它**不是一个独立的 web 服务器**，而是由我们 1.4 节反复提到的那个 **Gateway 常驻进程顺带 serve 出来的**——Gateway 在同一个端口（默认 `18789`）上，既跑 WebSocket、又把这个浏览器界面的静态页面发给你。所以你在浏览器里打开的那个对话框，背后是通过 WebSocket 直连 Gateway 的。这就引出一个和上一节正好成对照的关键点：**`tui --local` 是"绕过 Gateway"的路，而 Control UI 恰恰是"走 Gateway"的那条路**。还记得 1.4 节那张拓扑图里，Gateway 中枢旁边用绿色虚线分叉出的 `--local` 旁路吗？这一节我们要走的，就是回到中枢、从 Gateway 正门进来的主路。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>两条路对比：tui --local（1.4 节）× 浏览器 Control UI（本节）</font></p>
<div class="center">

| 对比维度 | `tui --local`（1.4 节） | Control UI（本节） |
|----------|------------------------|--------------------|
| 入口命令 | `pnpm openclaw tui --local` | 浏览器打开 `http://127.0.0.1:18789/`（或 `pnpm openclaw dashboard`） |
| 和 Gateway 的关系 | 绕过 Gateway，起进程内嵌运行时 | 依赖 Gateway，由它在 `18789` 端口 serve |
| 界面形态 | 终端全屏 TUI | 浏览器图形界面 + 对话框 |
| 额外前置 | 无（配好 provider 即可） | 需要 Control UI 资产（Gateway 会自动构建，建议首次手动 `pnpm ui:build`） |
| 适合场景 | 快速验证、贴源码读机制 | 图形化日常操作、多面板管理 |

</div>

&emsp;&emsp;道理讲清楚了，我们动手把它打开。因为 Control UI 依赖 Gateway，所以打开它要比上一节多两个前置动作：先把浏览器界面的静态资源构建出来，再把 Gateway 跑起来，最后用一条命令打开浏览器。下面分三步走。

> 🔥 **踩坑预警 · 本节命令请在外部终端执行**：这一节的 `pnpm openclaw gateway` 是一个**常驻进程**（跟 1.4 节的 tui 一样会一直占着终端不返回），`pnpm openclaw dashboard` 则会**直接拉起你的浏览器**——这两类都不适合在 Jupyter Notebook 里跑（gateway 会让 cell 一直挂在 `[*]` 状态）。下面的命令格只是把"该敲什么"原样列给你，**请复制到真实终端里执行**（macOS Terminal / Linux shell / WSL shell），跑法和上一节那条会挂起的 tui 命令一样。

**步骤一：构建浏览器界面的静态资源**

&emsp;&emsp;Control UI 默认就是开启的，浏览器界面的静态资源放在 `dist/control-ui` 目录里。这里有个贴心的设计：在源码 checkout 下，如果 Gateway 启动时发现这份资源还没构建，它会**自动尝试帮你构建**；万一自动构建也没成功，你访问页面时会看到一段 `503` 的"资源未就绪"提示（而不是莫名其妙的白屏）。话虽如此，我们仍**建议首次手动构建一次**——一来能把构建问题提前暴露在这一步，而不是等访问时才发现；二来构建好后首次打开更快，不必等运行时现场构建。手动构建在仓库目录里用 `pnpm ui:build` 完成，它会把界面编译进 `dist/control-ui/`，只需在首次使用（或界面源码更新后）跑一次。

In [ ]:
# 首次构建 Control UI 静态资源（跑完即返回）；Gateway 启动时也会自动尝试构建
# 手动先跑能提前暴露构建问题，产物落到 dist/control-ui/
!cd openclaw && pnpm ui:build

&emsp;&emsp;跑完你会看到 `Vite` 的构建进度，结束后 `dist/control-ui/index.html` 就生成了。可以用 `ls dist/control-ui/` 确认它确实存在——有了它，下一步 Gateway 就能直接把界面发给浏览器，省去运行时再现场自动构建的那点等待。

**步骤二：启动 Gateway 常驻进程**

&emsp;&emsp;静态资源就位后，就该把真正 serve 它的 Gateway 跑起来了。前面说过，Gateway 是那个"一直在后台接消息、跑 agent"的常驻进程，Control UI 就挂在它身上。我们用 `pnpm openclaw gateway` 启动它（源码里这条命令的职责描述是 "Run, inspect, and query the WebSocket Gateway"），它起来后会监听默认的 `18789` 端口。

In [ ]:
# 启动 Gateway 常驻进程（它会一直运行、监听 18789，不会自动退出）
# ⚠️ 这是常驻进程，请在外部终端执行；在 Notebook 里跑会一直挂起
!cd openclaw && pnpm openclaw gateway

&emsp;&emsp;这条命令跑起来后终端不会返回，而是停在那里持续运行——这是**正常的**，说明 Gateway 正在在线待命，不要因为它"没退出"就以为卡住了。开发时如果你希望改完源码自动重启，也可以用仓库提供的 `pnpm gateway:watch`（带热重载）。把这个终端窗口留着别关，我们另开一个新终端做下一步。

**步骤三：打开浏览器控制台**

&emsp;&emsp;Gateway 在线后，打开 Control UI 有两种方式：要么直接在浏览器地址栏访问 `http://127.0.0.1:18789/`，要么用 OpenClaw 给你准备的快捷命令 `pnpm openclaw dashboard`。后者更省事——它会把链接复制到剪贴板、并直接拉起浏览器；如果你的 token 是本地直存的（而不是交给外部 SecretRef 托管的），它还会顺手把 token 一起拼进链接，免去你手动粘贴（源码里这条命令的描述就是 "Open the Control UI with your current token"）。

In [ ]:
# 打开浏览器控制台（自动带 token、自动拉起浏览器）
# 等价于手动访问 http://127.0.0.1:18789/ ；只想拿链接不开浏览器可加 --no-open
!cd openclaw && pnpm openclaw dashboard

&emsp;&emsp;命令跑完，你的默认浏览器就会弹出 OpenClaw 的控制台界面。如果它顺利连上 Gateway，你会看到下面这样一个完整的图形界面——这就是我们这一节的目标，一个真正能用对话框跟 agent 聊天的前端看板：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102207004.png" width=80%></div>

&emsp;&emsp;这张界面信息量不小，我们对着它把几个关键区域认一遍，你就知道它和前面讲的机制是怎么对应上的。**左上角**是 OpenClaw 的红龙虾 logo 和版本号（图中是 `v2026.5.28`），左下角那个绿色小圆点表示 Gateway 的 WebSocket 已经连通——这是判断"有没有连上"最直接的信号。**中间最大的区域**就是"聊天"标签页，也就是这一节的主角对话框：图中我们发了一句"你好"，agent（图中名叫 Nova）回复了"你好"，这一来一回走的正是 Gateway 的 `chat.send` / `chat.history` 接口，跟 tui 里发消息是同一套会话和路由规则。**顶部那一排下拉框**尤其值得多看一眼——`main` 是当前 agent、`Main Session` 是当前会话、`DeepSeek` 是当前正在用的模型供应商：还记得第 0 章讲"龙虾是跨 model provider"的吗？那个抽象的三维定位，此刻就具体成了你能在这个下拉框里随手切换的 provider。**左侧那一列导航**（概览 / 活动 / 工作板 / 实例 / 会话 / 使用情况 / 定时任务 / 文档）则是 Control UI 的完整面板集——chat 只是其中一个标签，它还能管会话、看活动、配定时任务。**底部输入框**旁边的"开始 Talk"是浏览器里的实时语音对话入口，这里先知道有这么个东西即可。

> 📌 **为什么本机能直接打开、没让你"配对"**：Control UI 是一个能改配置、能批准 exec 的**管理界面**，所以默认对新设备会要求一次性配对审批（你可能会撞见 `1008: pairing required` 这个提示）。但有一条豁免：**从本机回环地址（`127.0.0.1` / `localhost`）发起的浏览器连接，这道"设备配对"会被自动批准**（要分清：它免的只是"配对审批"这一步，Gateway 的 token 鉴权仍然照常生效），所以你在自己电脑上按上面三步打开时不会被配对拦住。只有当你换成用局域网 IP 或 Tailscale 从**另一台设备**访问时，才需要在 Gateway 这边用 `pnpm openclaw devices list` 看待批请求、再用 `pnpm openclaw devices approve <requestId>` 批准。

> 🔥 **踩坑预警 · 页面提示资源未就绪（503）**：如果浏览器能打开、但显示一段 `503` 和"Control UI 资源不可用"之类的提示，说明 `dist/control-ui` 资产缺失、而且 Gateway 的自动构建这次也没成功（常见于依赖没装全、或构建过程报了错）。后果是界面出不来，但这不是命令失败、也不是白屏，而是资产没就绪。正确做法：回到步骤一手动跑一次 `pnpm ui:build`，看构建过程具体报什么错并解决。排查方法：先 `ls openclaw/dist/control-ui/index.html` 确认这个文件在不在，不在就说明资产确实没构建出来。

> 🔥 **踩坑预警 · 提示 unauthorized / 1008**：如果界面提示鉴权失败，多半是 token 没带上。Gateway 默认开启了共享密钥鉴权，`pnpm openclaw dashboard` 通常会自动把 token 拼进链接；万一没生效，可以用 `pnpm openclaw config get gateway.auth.token` 把 token 取出来，粘进 Control UI 设置面板的鉴权框里再连。排查方法：如果你不确定到底是 token 问题、还是 Gateway 压根没起来，先用 `pnpm openclaw status` 确认 Gateway 在线（官方排障也是用它做可达性检查），排除"其实是步骤二没跑成功"这种情况，再回头查 token。

&emsp;&emsp;为了方便你回头核对，这一节用到的命令和端口都能在源码与官方文档里查到：默认端口 `18789` 定义在源码 `src/config/paths.ts:263`（常量 `DEFAULT_GATEWAY_PORT`）；`dashboard` 子命令注册在 `src/cli/program/register.maintenance.ts:93`（`.command("dashboard")`）；`gateway` 子命令注册在 `src/cli/gateway-cli/register.ts:457`（`.command("gateway")`）；Control UI 的完整能力清单与鉴权说明则在仓库文档 `docs/web/control-ui.md` 与 `docs/web/dashboard.md` 里。

&emsp;&emsp;到这里，你就把 OpenClaw 的**两条路**都走通了：一条是 1.4 节的 `tui --local`，在终端里、绕过 Gateway、单机直跑，适合快速验证和读源码对照；另一条是这一节的 Control UI，走 Gateway 正门、在浏览器里图形化操作，把第 0 章那个抽象的"跨 agent × 跨 provider × 多面板"定位变成了你眼前可点可切的看板。两条路各有各的用处，后面章节我们主要还是沿 `tui --local` 这条"贴近源码"的路深入机制，但你心里要清楚另一条图形化的路一直都在。环境的两条路都打通了，下面我们就把日常最顺手的几招玩法集中过一遍。

### 1.6 上手就能玩：TUI 里最常用的几招

&emsp;&emsp;第一条消息只是开胃菜。OpenClaw 跑起来之后，日常最常打交道的就是三类东西：在交互界面里用斜杠命令快速控场、给 agent 装上现成的技能包（skills）、以及用一句话指挥它干各种真实的活。我们一个个来——它们的体感和 Claude Code 很像，你要是用过会很快上手。

&emsp;&emsp;先说一个**关于怎么跑**的要紧前提，免得你在 Notebook 里白等。这一节的命令分两类：一类是进了 `tui` 交互界面之后**在里面敲**的（斜杠命令、对话消息），它们得在真实终端的 tui 界面里输入——1.4 节那个"tui 在 Notebook 里会挂起"的坑这里同样适用；另一类是普通的 `openclaw` 子命令（比如查 skills 列表），跟 `--version` 一样跑完就返回，可以直接在 Notebook 里用 `!` 执行。下面每处都会标清楚是哪一类。

&emsp;&emsp;**案例一：斜杠命令——TUI 里的快捷控制台**

&emsp;&emsp;进入 tui 界面后，凡是以 `/` 开头的输入都不是发给模型的消息，而是直接控制这次会话的快捷命令——这点和 Claude Code 的斜杠命令几乎一模一样。最常用的有这么几个：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>TUI 常用斜杠命令速览（在 tui 交互界面内输入）</font></p>
<div class="center">

| 斜杠命令 | 作用 |
|----------|------|
| `/help` | 列出当前可用的所有斜杠命令 |
| `/status` | 查看 gateway、会话、上下文占用等状态 |
| `/model <provider/model>` | 切换当前使用的模型（或 `/models` 打开选择器） |
| `/think <级别>` | 调节模型的思考强度 |
| `/usage <off\|tokens\|full>` | 切换每条回复后的 token 用量显示 |
| `/agent <id>` | 切换到另一个 agent（或 `/agents` 打开选择器） |
| `/new`（或 `/reset`） | 清空当前会话、重新开始 |
| `/abort` | 中断正在执行的任务 |
| `/exit`（或 `/quit`） | 退出 tui |

</div>

&emsp;&emsp;这些命令都在 tui 界面里直接敲，不是 Notebook 里的代码。举个例子，你进了界面想换个模型、再看下用量和状态，会这样输入（注意这是 tui 内部输入，不是终端命令）：

```text
/model <provider/model>    # 换成你在 1.3 配好的某个模型
/usage tokens
/status
```

&emsp;&emsp;它们的共同点是即时生效、不消耗对话轮次——你可以把它们理解成 tui 的"控制面板"。

&emsp;&emsp;**案例二：skills——给 agent 装上现成的"技能包"**

&emsp;&emsp;skills 是 OpenClaw 里一个特别实用的能力：把某一类专门的本领（比如操作 Apple 备忘录、读写提醒事项、调用 1Password）打包成一个可复用的"技能包"，agent 需要时就调用它。这正对应你可能在 Claude Code 里用过的 skills——OpenClaw 的玩法很相似，而且这套命令是普通子命令，**不依赖 Gateway，在本节 `--local` 环境下就能直接跑**。先看看你这台机器上现在有哪些 skills 可用：

In [ ]:
# 查看本机可用的 skills（普通子命令，跑完即返回，可在 Notebook 直接执行）
!cd openclaw && pnpm openclaw skills list

&emsp;&emsp;跑完你会在顶部看到一行汇总，类似 `Skills (N/M ready)`（N 是已就绪可用的数量、M 是识别到的总数），下面跟着一张表，逐个列出 skill 的状态、名字、用途和来源。状态主要分两种：`✓ ready`（已就绪，可直接用）和 `△ needs setup`（还要装个命令行工具或做点配置才能用）；来源里 `openclaw-bundled` 表示是 OpenClaw 自带的。

> 🔥 **踩坑预警 · skills list 的提示信息**：你的输出里可能混着一些提示，比如某些 skill 标着 `△ needs setup`（它依赖的外部工具还没装），或者一串关于路径被跳过的提示。前者很正常——按提示装上对应工具即可；后者多半是你本机 skills 目录有特殊的软链接配置，OpenClaw 出于安全把越界路径跳过了，不影响 list 本身。看汇总行和表格主体就够了。

&emsp;&emsp;光看列表不过瘾，skills 真正的用法是装新的、再让 agent 用。常用的几条命令是：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>skills 常用命令（普通子命令，在仓库目录内用 pnpm openclaw 执行）</font></p>
<div class="center">

| 命令 | 作用 |
|------|------|
| `pnpm openclaw skills search "关键词"` | 在官方市场 ClawHub 搜索 skill |
| `pnpm openclaw skills install <名称>` | 从 ClawHub 安装一个 skill（会联网下载） |
| `pnpm openclaw skills install git:用户/仓库` | 从 Git 仓库安装 |
| `pnpm openclaw skills info <名称>` | 查看某个 skill 的详细信息 |

</div>

&emsp;&emsp;装好之后，你不用记什么特殊语法——直接在 tui 里用大白话让 agent 干活就行，它会自己判断要不要调用对应的 skill。你甚至可以让它现场帮你**造**一个新 skill，比如在 tui 界面里说：

```text
Make a skill called morning-catchup that runs my Monday inbox routine.
（帮我做一个叫 morning-catchup 的 skill，执行我周一的收件箱例行流程）
```

&emsp;&emsp;这句话会触发 OpenClaw 的 skill 创建流程，由 agent 引导你把这个技能包搭起来。注意这属于 tui 交互界面里的对话，不是 Notebook 命令。

&emsp;&emsp;**案例三：经典任务 prompt——看 agent 干不同的活**

&emsp;&emsp;说到底，OpenClaw 最核心的价值是能调动工具链干真实的活，而不只是聊天。1.4 节"查一下现在几点，把结果写进 a.txt"已经让你见识了一次（agent 会自主调用工具完成它，具体调几个由模型决定）。这里再给你几条经典任务，每条都会逼 agent 走一条不同的工具链，你可以在 tui 界面里挨个试：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>几条经典上手任务（在 tui 界面输入，或用 --message 单发）</font></p>
<div class="center">

| 你对 agent 说的话 | 大致会触发的工具链 |
|-------------------|--------------------|
| 查一下现在几点，把结果写进 a.txt | agent 自主调用工具（如一条 `exec` 直接写文件，或 `exec`/`write` 两步） |
| 读一下当前目录的 README.md，用三句话总结 | 读文件（read）→ 模型总结 |
| 看看这台机器磁盘还剩多少空间 | 执行 df 命令（exec）→ 解读输出 |
| 在当前目录建一个 hello.py，打印今天日期 | 写文件（write）→ 可选跑一下验证（exec） |

</div>

&emsp;&emsp;这几条的共同点是：你说的都是大白话目标，没指定"用哪个工具"，是 agent 自己拆解成一串工具调用去完成的——这恰恰是 agent 和普通聊天机器人的本质区别。提醒一下，这些任务要么在 tui 交互界面里输入，要么像上一节那样用 `pnpm openclaw tui --local --message "…"` 单发；两种都会进入 tui，所以在 Notebook 里同样可能挂起，想顺畅体验就在真实终端里跑。

&emsp;&emsp;这三招——斜杠命令控场、skills 装技能、用大白话派任务——够你把这只龙虾使唤得团团转了。而你大概也会越用越好奇：每次你敲下一句话，agent 内部到底经历了什么，才把它变成一串工具被一个个亮出来、结果一个个回填？这个"黑箱"正是下一章要打开的。下面我们就钻进 agent 的"心脏"，看那个 while 循环和事件流，是怎么把你这句话变成真实行动的。

---

## <center>第 2 章：agent 运行时心脏——while 循环 × EventStream × steering</center>

&emsp;&emsp;上一章你已经在 TUI 上亲眼看到 OpenClaw 调用工具、把活干完了，而且我们在章末抛出了那个反直觉的命题：OpenClaw 跑到一半能被插话。这一章我们就把这件事的来龙去脉彻底讲清。我们会先做一次认知重塑，弄明白"能插话"为什么是 agent 的本质特征（第 2.1 节）；然后向内挖，看支撑这件事的双层 while 循环到底长什么样（第 2.2 节）；接着看这个循环对外吐出的事件流——10 种事件、4 组生命周期的完整谱系（第 2.3 节）；最后回到真实工具里，用 OpenClaw 自带的可观测面把我们讲的机制对一遍账（第 2.4 节），并在第 2.5 节确认你这一章到底学到了什么。

&emsp;&emsp;这一章的核心可以浓缩成三句话：**第一，agent 是一个"可中断、可引导"的持续循环，不是一次性的请求—响应；第二，这个循环用一个外层 while 维持心跳、用一个内层 while 处理单轮内的所有工具调用和插话；第三，循环的每一步状态变化都以事件的形式 emit 出来，构成一条可观测的事件流。** 把这三句话和下面的源码锚点对上，你就读懂了 OpenClaw 的 agent-core 内核。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102200324.png" width=70%></div>

### 2.1 认知重塑：你以为的 AI 助手 vs 真正的 agent

&emsp;&emsp;我们先停下来想一个场景。假设你给一个聊天机器人发了个慢任务："帮我逐行读完这个一千行的大文件，然后总结。" 它开始干活了。这时你突然意识到自己只想要前十行的摘要。你心里那个本能反应大概是："算了，等它跑完我再重新发一遍吧"——因为在 chatbot 的世界里，**一次对话是原子的、不可打断的**。你发出去的那一刻，这一轮就锁死了，要么等它结束，要么放弃重来。

&emsp;&emsp;这正是我们要打碎的认知。在 OpenClaw 这样的 agent 运行时里，你完全可以**趁它还在跑的时候**再发一句"改成只读前 10 行"，而这句话不会另起一个新会话，而是**插进它当前正在跑的这一轮**，让它中途调整。这个能力有一个专门的名字，叫 **steering**（引导/操舵）。

&emsp;&emsp;为什么这件事这么重要，值得我们单独拎出来讲？因为它揭示了 agent 和 chatbot 在架构上的根本区别。下面这张表把两者并排放一起，你对照着看。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>chatbot 与 agent 的架构分水岭</font></p>
<div class="center">

| 维度 | 你以为的 AI 助手（chatbot） | 真正的 agent（OpenClaw） |
|------|------------------------------|---------------------------|
| 交互模型 | 一问一答，每轮对话独立 | 持续循环，跨轮维持状态 |
| 运行中能否插话 | 不能，本轮锁死 | 能，steering 插入当前轮 |
| 本质结构 | 请求—响应函数 | 可中断、可引导的循环 |
| 插话的归宿 | （只能等下一轮）| 注入正在跑的这一轮，不另起会话 |

</div>

&emsp;&emsp;表里最关键的一行是"运行中能否插话"。OpenClaw 的源码在 agent loop 的开头就埋了一个动作——去检查"用户是不是趁等待的时候又输入了新消息"。这个检查对应的真实代码在 `packages/agent-core/src/agent-loop.ts` 第 213 行：`config.getSteeringMessages?.()`。注意它前一行（第 212 行）有一句注释，原文是 "Check for steering messages at start (user may have typed while waiting)"——OpenClaw 的作者把这件事写得明明白白。

&emsp;&emsp;这里有一个特别容易踩的误区，我们提前预警。

> **【常见误区】**：以为 steering 是"另起一段对话"。后果是你会把 agent 理解成"能连续聊天的 chatbot"，从而完全错过它"运行中可引导"这个本质能力。正确理解是：steering 消息插入的是**当前正在跑的同一轮 loop**，agent 会在不结束本轮的前提下吸收你的新指令。但要避免过度理解：steering **不会打断正在执行的工具调用**——它是在当前这批工具跑完、本轮 `turn_end` 之后、下一次模型调用之前才被注入的（源码 `types.ts:222` 注释原文："Tool calls from the current assistant message are not skipped"）。所以"运行中插话"严格说是"在每轮的模型调用边界（model boundary）插话"，而非把正在跑的工具流式打断。排查方法：在 OpenClaw 的 TUI 界面里发一个慢任务，趁它运行时再发一句，你会看到新消息出现在当前任务流里，而不是新开一个对话块。

&emsp;&emsp;到这里，认知已经被重塑了：agent 的心脏不是一个函数，而是一个**循环**，而且是一个会主动"回头看你有没有新话要说"的循环。下一节我们就把这个循环的骨架剖开。

### 2.2 while 双层结构剖析

&emsp;&emsp;既然 agent 的心脏是个循环，那它具体长什么样？我们打开 `agent-loop.ts`，会发现它不是一个简单的 `while`，而是**两层嵌套的 while**。这个双层结构是理解整个 agent-core 的钥匙，我们一层一层看。

&emsp;&emsp;外层是 `while (true)`，对应源码第 216 行。它的职责是**维持心跳**——只要 agent 这次生命周期还没结束，外层循环就一直转。什么时候会重新进入外层循环的下一圈？当内层把这一轮的工具调用和插话都处理完、agent 本来要停下时，源码会再查一次有没有新的**后续消息**（follow-up，对应第 315 行的 `getFollowUpMessages`）——有就再转一圈把它接住。这里要和 steering 分清楚：**steering 是让内层循环继续转**（每轮在第 311 行取一次、不结束当前这一轮），而**让外层循环重启一圈的是 follow-up**（当前轮已经收尾之后又冒出来的新消息）。

&emsp;&emsp;内层是 `while (hasMoreToolCalls || pendingMessages.length > 0)`，对应源码第 220 行。它的职责是**处理单轮内的所有事情**——既包括模型这一轮想调用的所有工具（`hasMoreToolCalls`），也包括用户中途插进来的 steering 消息（`pendingMessages`）。只要这两样里还有没处理完的，内层循环就继续转。

&emsp;&emsp;还有两个关键的钩子要认识。一个是我们上节见过的 `getSteeringMessages`（第 213 行），它在 loop 开头和每轮结束时（第 311 行）各取一次 steering 消息——这就是"回头看你有没有新话要说"的具体落点。另一个是 `prepareNextTurn`，对应源码第 285 行的 `config.prepareNextTurn?.(nextTurnContext)`，它让 agent 在每轮结束前有机会换掉 context、换模型或调整推理配置。

&emsp;&emsp;注意 `prepareNextTurn` 后面那个 `?.`——这是 TypeScript 的可选链调用，意味着这个钩子是**可选的**，不是每轮必然发生。我们在等会的 MVP 里会用 `if callback:` 来模拟这个"可选"语义。

&emsp;&emsp;光读源码描述还不够，下面你直接用 Python 把这个双层 while 重现出来跑一遍。下面这段代码是本章所有 MVP 的共享地基——它用标准库的 `dataclass` 和 `enum` 定义事件类型和循环结构，运行后你会看到一条完整的事件序列被打印出来。如果序列首尾不是 `agent_start` / `agent_end`，说明循环结构错了。

&emsp;&emsp;这段代码偏长（一百多行），先给你一条阅读路径：它由两部分组成——前半部分（`EventType` 枚举、`Event` 和 `MiniAgentLoop` 这几个 `dataclass`）是机制的"零件定义"，你不需要一次读透；真正要跑通看效果的入口在后半部分的 `loop.run(...)` 调用和底部的断言。建议你第一遍**先直接运行整段、看断言输出**，确认"它真能跑出 agent_start→...→agent_end 的序列"，第二遍再回头对照注释逐行理解类定义。另外提醒一句：这个 cell 是本章后面 Cell B、Cell D 共享的依赖，它们都用到这里定义的 `MiniAgentLoop` 和 `EventType`，所以**请确保先运行本 cell 再运行后续 cell**；如果你重启过 kernel 或跳着跑，遇到 `NameError` 多半就是漏跑了这一格。

In [6]:
# Python 最小重现，非 OpenClaw 真实源码（真实实现见 TS: packages/agent-core/src/agent-loop.ts）
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional

# ── 事件类型：对照真实源码 types.ts:410-437 的 AgentEvent（共 10 种，分 4 组）──
# 本 MVP 聚焦贯穿全程的 6 种主时序骨架（agent/turn/message 生命周期），
# 工具执行三件套 tool_execution_start/update/end 在 Cell B 演示，凑齐完整 10 种。
class EventType(str, Enum):
    """AgentEvent 的事件类型，名字与真实源码 emit 的 type 字段一致。

    源码全集 10 种（types.ts:410-437），按生命周期分 4 组；
    此处定义 6 种主骨架 + 工具执行三件套（start/update/end），message_update（流式增量）本 MVP 略去。
    """
    # Agent lifecycle
    AGENT_START           = "agent_start"
    AGENT_END             = "agent_end"
    # Turn lifecycle
    TURN_START            = "turn_start"
    TURN_END              = "turn_end"
    # Message lifecycle
    MESSAGE_START         = "message_start"
    MESSAGE_END           = "message_end"
    # Tool execution lifecycle 三件套（Cell B 演示 start→update→end）
    TOOL_EXECUTION_START  = "tool_execution_start"
    TOOL_EXECUTION_UPDATE = "tool_execution_update"
    TOOL_EXECUTION_END    = "tool_execution_end"

@dataclass
class Event:
    """一个 emit 出来的事件。

    Args:
        type: 事件类型（AgentEvent 10 种之一）
        note: 附注，记录 tool 名 / steering 标记等便于肉眼对账
    """
    type: EventType
    note: str = ""

@dataclass
class MiniAgentLoop:
    """agent loop 的最小重现：双层 while + steering 队列 + 事件流。

    Args:
        steering_queue: steering 待插消息队列（模拟用户运行中插话）
        events: 收集 emit 出的事件序列，供肉眼对账
        prepare_next_turn: 可选钩子，对应 config.prepareNextTurn?.()
    """
    steering_queue: list = field(default_factory=list)
    events: list = field(default_factory=list)
    prepare_next_turn: Optional[Callable] = None

    def emit(self, etype: EventType, note: str = ""):
        # 把一次状态变化 emit 成事件，append 进序列（真实源码走 EventStream 总线）
        self.events.append(Event(etype, note))

    def get_steering_messages(self):
        # 取并清空 steering 队列。两处调用：loop 开头取对应 :213（启动时已排队的消息）、
        # 内层每圈 turn_end 之后取对应 :311（运行中插话）。真实源码这两处都调 getSteeringMessages?.()
        msgs, self.steering_queue = self.steering_queue, []
        return msgs

    def run(self, initial_prompt: str, scripted_tool_calls: list, steer_after_turn: int = None, steer_msg: list = None):
        # scripted_tool_calls：预设每轮是否还有 tool call（模拟模型决策，元素=工具名）
        self.emit(EventType.AGENT_START)
        self.emit(EventType.TURN_START)
        self.emit(EventType.MESSAGE_START, f"user: {initial_prompt}")
        self.emit(EventType.MESSAGE_END,   f"user: {initial_prompt}")

        # ── 外层 while(true)：持续循环维持心跳，对应 agent-loop.ts:216 ──
        pending = self.get_steering_messages()
        turn_idx = 0
        remaining = list(scripted_tool_calls)
        while True:
            has_more_tool_calls = len(remaining) > 0
            # ── 内层 while：处理本轮所有 tool call + steering，对应 agent-loop.ts:220 ──
            while has_more_tool_calls or len(pending) > 0:
                turn_idx += 1
                if turn_idx > 1:
                    self.emit(EventType.TURN_START, f"turn#{turn_idx}")  # 非首轮重新 emit，对应 :222

                # steering 消息插入当前轮（不另起会话），对应 :230 注入
                for s in pending:
                    self.emit(EventType.MESSAGE_START, f"steering: {s}")
                    self.emit(EventType.MESSAGE_END,   f"steering: {s}")
                pending = []

                if has_more_tool_calls:
                    tool = remaining.pop(0)
                    # 工具执行三件套 start→update→end，对应源码 tool_execution_start(:484/545)、
                    # tool_execution_update(:735 发 partialResult)、tool_execution_end(:813)
                    self.emit(EventType.TOOL_EXECUTION_START,  f"tool={tool} args=...")
                    self.emit(EventType.TOOL_EXECUTION_UPDATE, f"tool={tool} partialResult=...")  # 中间态：流式回报进度
                    self.emit(EventType.TOOL_EXECUTION_END,    f"tool={tool} result=ok isError=False")
                    # turn_end 携带 toolResults 字段（不是独立事件），对应 :277
                    self.emit(EventType.TURN_END, f"toolResults=[{tool}]")
                else:
                    self.emit(EventType.TURN_END, "toolResults=[]")

                # ── 每个 turn_end 之后 drain steering（对应真实 :311，在内层每圈末，不是内层外）──
                # 先模拟"此刻用户插话"：到约定轮次把消息入队（真实里就是用户在 turn_end 后敲键盘那一刻）
                if turn_idx == steer_after_turn and steer_msg:
                    self.steering_queue = list(steer_msg)
                pending = self.get_steering_messages()  # 轮末取 steering，对应 :311
                has_more_tool_calls = len(remaining) > 0

            # 内层退出（本轮无工具、无新插话）。每轮结束前可选换 context/model，对应 :285（?. 即可选）
            if self.prepare_next_turn:
                self.prepare_next_turn(turn_idx)
            # MVP 不演示 followup（:315）；无新插话即收束
            break

        self.emit(EventType.AGENT_END)
        return self.events

# ── Cell A 自检：双层 while 基线（无 steering）──
loop = MiniAgentLoop()
evs = loop.run("查一下现在几点，把结果写进 a.txt", scripted_tool_calls=["exec", "write"])
print(">>> Cell A 基线事件序列（无 steering）：")
for e in evs:
    print(f"  {e.type.value:<14} {e.note}")
seq = [e.type for e in evs]
# Tier 1 结构断言：首尾必须是 agent_start/agent_end，且循环至少跑出一个 turn_end
assert seq[0] == EventType.AGENT_START and seq[-1] == EventType.AGENT_END, "首尾必须是 agent_start/agent_end"
assert EventType.TURN_END in seq, "双层 while 必须至少产出一个 turn_end"
print("[OK] Cell A 断言通过：首 agent_start / 尾 agent_end / 含 turn_end")

>>> Cell A 基线事件序列（无 steering）：
  agent_start    
  turn_start     
  message_start  user: 查一下现在几点，把结果写进 a.txt
  message_end    user: 查一下现在几点，把结果写进 a.txt
  tool_execution_start tool=exec args=...
  tool_execution_update tool=exec partialResult=...
  tool_execution_end tool=exec result=ok isError=False
  turn_end       toolResults=[exec]
  turn_start     turn#2
  tool_execution_start tool=write args=...
  tool_execution_update tool=write partialResult=...
  tool_execution_end tool=write result=ok isError=False
  turn_end       toolResults=[write]
  agent_end      
[OK] Cell A 断言通过：首 agent_start / 尾 agent_end / 含 turn_end


&emsp;&emsp;这段代码做的事，就是把上面读到的双层 while 骨架原样搭出来。运行后你会看到两个工具（`exec` 取时间、`write` 写文件）各自对应一次 `turn_end`，中间夹着一个非首轮的 `turn_start`——这正说明"一次 agent 生命周期里可以有多个 turn"。（这里要说明一点：我们的 MVP 故意用 `exec`、`write` 两个工具，是为了把"多 turn"演示清楚；真实模型跑同样这条任务，可能用一条 `exec` 一步就完成，工具数会不一样，但"一次生命周期里有多个 turn"这个机制是一致的。）这里需要强调的是，我们在 `prepare_next_turn` 上用 `if self.prepare_next_turn:` 来模拟源码里 `?.` 的可选语义：钩子没传就不调用，避免把它理解成"每轮必然换 context"。

&emsp;&emsp;再点破一个关键简化：这个 MVP 让模型每次响应**只调一个工具**，所以你看到"一个工具对应一个 `turn_end`"。但真实 OpenClaw 源码不是这样——一个 turn 处理的是模型这次响应里要求的**整批**工具调用：`agent-loop.ts:256` 会把该 assistant 消息里所有 `toolCall` 一起取出再交给 `executeToolCalls` 执行，可能多个并行或串行。所以准确说：**一个 turn = 模型的一次响应，可含 0 个、1 个或多个工具**——turn 的数量取决于"模型开了几次口"，而不是"调了几个工具"。

&emsp;&emsp;下一节我们把注意力从循环骨架移到它吐出的事件流上。

> **【Tier 1 验证已通过】**：上面的断言独立验证了双层 while 结构的两个共性——序列首尾闭合（agent_start → agent_end）、循环至少产出一个 turn_end。这是"零件能用"的结构性证据。

### 2.3 EventStream 事件流——10 种事件 · 4 组生命周期

&emsp;&emsp;上一节我们把循环骨架搭起来了，但你可能注意到一个细节：循环里每走一步，我们都在调 `emit(...)`。这个 `emit` 不是随手打印，它对应 OpenClaw 里一个核心设计——**EventStream**。在真实源码里，agent-core 自带一条流式事件总线，loop 的每一次状态变化都以"事件"的形式发到这条总线上，外部（比如 TUI 界面）订阅这条流就能实时知道 agent 此刻在干什么。这正是 OpenClaw 的 TUI 能像 Claude Code 那样"工具名一个个亮出来"的底层原因。

&emsp;&emsp;那么这条流里到底会流出哪些事件？答案是确定的：`AgentEvent` 这个联合类型的定义就在 `packages/agent-core/src/types.ts` 第 410 到 437 行，去重后是**确切的 10 种事件**，源码把它们按职责分成了**4 组生命周期**。先把这张全集表放出来，你可以逐个核对——其中我们标了 `[主骨架]` 的 6 种，是贯穿一次 agent 运行的"主时序骨架"，后面的 MVP 重点演示它们；带工具/流式标记的另外 4 种则补全了完整谱系。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>AgentEvent 全集：10 种事件 · 4 组生命周期（含源码锚点）</font></p>
<div class="center">

| 生命周期组 | 事件名 | 含义 | 源码锚点 |
|------------|--------|------|----------|
| Agent lifecycle | `agent_start` [主骨架] | 一次 agent 生命周期开始 | types.ts:412；emit 第 122 / 152 行 |
| Agent lifecycle | `agent_end` [主骨架] | 一次 agent 生命周期结束 | types.ts:413；emit 第 194 / 251 / 307 / 326 行 |
| Turn lifecycle | `turn_start` [主骨架] | 一轮处理开始（每轮重新 emit）| types.ts:415；emit 第 123 / 153 / 222 行 |
| Turn lifecycle | `turn_end` [主骨架] | 一轮处理结束，**携带 message + toolResults 字段** | types.ts:416；emit 第 250 / 277 行 |
| Message lifecycle | `message_start` [主骨架] | 一条消息开始（user / assistant / toolResult 都发）| types.ts:418 |
| Message lifecycle | `message_update` | **仅 streaming assistant 消息**逐增量发射 | types.ts:420（注释在 419）；emit agent-loop.ts:397 |
| Message lifecycle | `message_end` [主骨架] | 一条消息结束 | types.ts:421 |
| Tool execution lifecycle | `tool_execution_start` | 工具开始执行，带 `{toolCallId, toolName, args}` | types.ts:423；emit 第 484 / 545 行 |
| Tool execution lifecycle | `tool_execution_update` | 工具执行中，带 `partialResult` | types.ts:424-430；emit 第 735 行 |
| Tool execution lifecycle | `tool_execution_end` | 工具执行结束，带 `{result, isError}` | types.ts:431-437；emit 第 813 行 |

</div>

&emsp;&emsp;这张表有四个点要划重点。第一，标 `[主骨架]` 的 6 种是主时序骨架——一次最简单的对话（无工具）只会流出这 6 种，所以下一节的 MVP 先聚焦它们把骨架讲透；但你要清楚**总共是 10 种**，工具执行和流式增量那 4 种不是遗漏，而是完整谱系的一部分。第二，`turn_start` 出现在多个行号——因为每一轮（不只是首轮）进入时都会重新 emit 一次，这呼应了上一节"一次生命周期多个 turn"的结构。第三，`turn_end` 里带着 `message` 和 `toolResults` 两个字段，意思是工具调用的**结果**是挂在 `turn_end` 上的；但工具执行的**过程**（开始/进行中/结束）则由独立的 `tool_execution_*` 三件套描述——这两者不要混。第四，`message_update` 是个容易踩的点：它**只**在流式输出 assistant 消息时逐增量发射（类型定义在 `types.ts:420`，紧邻它上一行 `types.ts:419` 的注释原文是 "Only emitted for assistant messages during streaming"），user 消息和 toolResult 消息都不会发它。

&emsp;&emsp;最后把 turn 和工具的**数量关系**钉死：一个 `turn`（一对 `turn_start`…`turn_end`）是模型的**一次响应**及其要求的那批工具执行，它**可以一次调用多个工具**，因此一个 turn 里可能 emit 好几组 `tool_execution_*` 三件套。上一节 MVP 为了把"多 turn"演示清楚，简化成了"一轮一个工具"，真实源码里一轮可含多工具——**别把 turn 数和工具数划等号**。

> **【打通到第 3 章】**：注意 `tool_execution_start` 携带的字段是 `{toolCallId, toolName, args}`——这组字段会原样流到界面层。第 3 章我们会看到 TUI 的 `chat-log.ts:333` 有个 `startTool(toolCallId, toolName, args)` 方法，参数名和这里**完全一致**。也就是说，第 3 章里 TUI"工具名一个个亮出来、参数逐个显示"的现象，底层正是这组 `tool_execution_*` 事件被订阅消费后渲染出来的。事件流（本章）和界面渲染（第 3 章）是同一条数据的两端。

> **【常见误区】**：以为 `turn_end` 和 `agent_end` 是一一对应、成对出现的。实际上一次 agent 生命周期可以有多个 `turn_end`（每个工具调用一轮、每次 steering 一轮），但只有一个 `agent_end` 收尾。后果是你会误以为"每结束一轮就该结束整个 agent"，从而读不懂多轮 loop。排查方法：看上节 Cell A 的输出，两个工具产出了两个 `turn_end`，但 `agent_end` 只出现一次。

&emsp;&emsp;光看表还不够直观，我们让事件流真的在眼前流一遍。下面这段在上节 MVP 基础上，故意往 steering 队列里塞一条消息。运行后你会看到完整的事件序列：既有 `message_start[steering]`（"插话被注入当前轮"的可视化证据），也有工具执行的 `tool_execution_start → tool_execution_update → tool_execution_end` 这一组三件套事件——这就是表里"工具执行 lifecycle"那一组在真实流转里长什么样。这一节我们把 10 种里能在最小模型里复现的那几种都跑出来。

&emsp;&emsp;在用代码跑之前，先用一张时序图把这条事件流的全景定格下来——看清从 `agent_start` 到 `agent_end` 事件按什么顺序流出、steering 插在哪一刻被注入当前轮、工具执行三件套怎么嵌在一轮之内，以及为什么一次生命周期可以有多个 `turn_end` 却只有一个 `agent_end`：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603171449171.png" width=70%></div>

In [7]:
# Python 最小重现，非 OpenClaw 真实源码（真实实现见 TS: packages/agent-core/src/agent-loop.ts）
# ── Cell B：含工具执行 + steering 插话的完整事件序列 ──
loop2 = MiniAgentLoop()
# 关键：不在 run 前预设队列（那样会在 loop 开头 :213 就注入，等于"启动前排队"）。
# 真实的"运行中插话"是某一轮 turn_end 之后才到达，这里用 steer_after_turn=1 模拟
# "第 1 轮工具跑完、turn_end 之后用户才插话"（对应源码 :311 的 drain 时机）。
evs2 = loop2.run("逐行读一个大文件并总结", scripted_tool_calls=["read"],
                 steer_after_turn=1, steer_msg=["改成只读前 10 行"])

print(">>> Cell B 事件序列（含 tool_execution 三件套 start→update→end + steering 插话）：")
for e in evs2:
    # steering 事件打 <== 标记；工具执行事件打 [tool] 标记，让两组事件肉眼可分
    if "steering" in e.note:
        tag = "  <== steering 在此插入当前轮"
    elif e.type.value.startswith("tool_execution"):
        tag = "  [tool] 工具执行 lifecycle"
    else:
        tag = ""
    print(f"  {e.type.value:<22} {e.note}{tag}")

# 看本次运行覆盖到了 10 种事件全集里的哪几种
seen = {e.type.value for e in evs2}
print("  本次覆盖的事件类型：", sorted(seen))
# 断言：工具执行 lifecycle 三件套 start/update/end 在本次确实齐发
assert {"tool_execution_start", "tool_execution_update", "tool_execution_end"} <= seen, \
    "调用工具的运行必须发出 tool_execution_start / update / end 三件套"
print("[OK] Cell B：tool_execution 三件套（start→update→end）齐发，工具执行 lifecycle 已体现")

>>> Cell B 事件序列（含 tool_execution 三件套 start→update→end + steering 插话）：
  agent_start            
  turn_start             
  message_start          user: 逐行读一个大文件并总结
  message_end            user: 逐行读一个大文件并总结
  tool_execution_start   tool=read args=...  [tool] 工具执行 lifecycle
  tool_execution_update  tool=read partialResult=...  [tool] 工具执行 lifecycle
  tool_execution_end     tool=read result=ok isError=False  [tool] 工具执行 lifecycle
  turn_end               toolResults=[read]
  turn_start             turn#2
  message_start          steering: 改成只读前 10 行  <== steering 在此插入当前轮
  message_end            steering: 改成只读前 10 行  <== steering 在此插入当前轮
  turn_end               toolResults=[]
  agent_end              
  本次覆盖的事件类型： ['agent_end', 'agent_start', 'message_end', 'message_start', 'tool_execution_end', 'tool_execution_start', 'tool_execution_update', 'turn_end', 'turn_start']
[OK] Cell B：tool_execution 三件套（start→update→end）齐发，工具执行 lifecycle 已体现


&emsp;&emsp;这段代码完成的是事件流的"可视化对账"。运行后你会看到 `agent_start → turn_start → message_start/end（初始消息）→ tool_execution_start/update/end → turn_end → turn_start → message_start/end（steering）→ turn_end → agent_end` 的流转：带 `<==` 标记的是 steering 消息——它出现在第 1 轮 `turn_end` 之后、第 2 轮开头（`turn_start` 之后），恰好对应源码 :311（每轮 turn_end 后 drain steering）和 :228（下一轮开头注入）——它被注入同一个 run 的下一轮，而非新开会话，也不会打断正在执行的工具；带 `[tool]` 标记的则是工具执行 lifecycle，工具结果最终又汇总进 `turn_end` 的 `toolResults` 字段。这一节我们把 10 种事件里能在最小模型复现的主骨架 + 工具执行组都跑出来看清了，下面进入本章的端到端验证。

&emsp;&emsp;到这里第 2.2、2.3 节的零件都已经各自验证过了，我们做一次本章的整机验证（Tier 2）——跑一次包含 steering 的完整 loop，断言事件序列的整体行为符合 `agent-loop.ts` 的 emit 语义。注意这次我们传入了一个 `prepare_next_turn` 钩子（空操作），用来确认"带钩子也能正常收束"。运行后你会看到断言全部通过；如果 `steering_idx` 不大于第 1 个 `turn_end` 的位置，说明 steering 被错误地放在了工具执行之前而非运行中（turn_end 之后）。

In [8]:
# Python 最小重现，非 OpenClaw 真实源码（真实实现见 TS: packages/agent-core/src/agent-loop.ts）
# ── Cell D（Tier 2 端到端）：含 steering + prepareNextTurn 钩子，断言整机行为 ──
def my_prepare(turn_i):
    """模拟 prepareNextTurn 钩子：每轮结束前换 context（此处空操作，仅验证钩子可被调用）。

    Args:
        turn_i: 当前轮序号
    """
    pass

loopD = MiniAgentLoop(prepare_next_turn=my_prepare)
# 运行中插话：第 1 轮 turn_end 之后用户补一句（对应 :311），下一轮带着新指令继续执行
evsD = loopD.run("读取并总结大文件", scripted_tool_calls=["read", "exec"],
                 steer_after_turn=1, steer_msg=["补充：只取前 10 行"])

seqD   = [e.type for e in evsD]
notesD = [e.note for e in evsD]

# 行为断言 1：序列首尾闭合
assert seqD[0]  == EventType.AGENT_START, "起始必须是 agent_start"
assert seqD[-1] == EventType.AGENT_END,   "终止必须是 agent_end"
# 行为断言 2：agent_end 全程只出现一次（一次生命周期一个收尾）
assert sum(1 for t in seqD if t == EventType.AGENT_END) == 1, "agent_end 必须唯一"
# 行为断言 3：steering 消息确实出现在序列中
assert any("steering" in n for n in notesD), "steering 消息必须出现"
# 行为断言 4：steering 在初始用户消息之后被注入（证明是插入当前会话，非另起会话）
init_msg_idx  = next(i for i, n in enumerate(notesD) if n.startswith("user:"))
steering_idx  = next(i for i, n in enumerate(notesD) if "steering" in n)
assert steering_idx > init_msg_idx, "steering 应在初始消息之后注入（插入当前会话）"
# 行为断言 5：steering 在第 1 个 turn_end 之后才注入（证明是"运行中插话"，非启动前排队/工具前插入）
first_turn_end_idx = next(i for i, t in enumerate(seqD) if t == EventType.TURN_END)
assert steering_idx > first_turn_end_idx, "steering 应在第 1 轮 turn_end 之后注入（运行中插话，对应 :311）"

print(">>> Cell D（Tier 2）端到端断言全部通过：")
print(f"   序列长度 = {len(evsD)}")
print(f"   steering 在第 {steering_idx} 位，晚于初始消息（第 {init_msg_idx} 位）→ 确认插入当前会话")
print("   完整序列：", [t.value for t in seqD])
print("[OK] 端到端：含 steering 的 loop 行为符合 agent-loop.ts emit 序列预期")

>>> Cell D（Tier 2）端到端断言全部通过：
   序列长度 = 16
   steering 在第 9 位，晚于初始消息（第 2 位）→ 确认插入当前会话
   完整序列： ['agent_start', 'turn_start', 'message_start', 'message_end', 'tool_execution_start', 'tool_execution_update', 'tool_execution_end', 'turn_end', 'turn_start', 'message_start', 'message_end', 'tool_execution_start', 'tool_execution_update', 'tool_execution_end', 'turn_end', 'agent_end']
[OK] 端到端：含 steering 的 loop 行为符合 agent-loop.ts emit 序列预期


&emsp;&emsp;这段端到端验证把本章的两条主轴（双层 while + EventStream 事件流）拧成一个整体跑了一遍：steering 注入位置正确、agent_end 唯一、序列首尾闭合。这就是"零件组装起来还能跑"的行为性证据。需要再次强调的是——**这段 Python 是为了让你亲眼看到机制本质的最小模型，OpenClaw 的真实 `agent-loop.ts` 比它复杂得多**（有流式增量、错误分支、并发 tool 执行等），我们刻意省略了那些工程细节，只保留循环和事件这两条主干。

### 2.4 章末收口

&emsp;&emsp;这一章我们从一个反直觉的命题出发——"agent 跑到一半你能插话"——一路挖到了它背后的双层 while 循环和 EventStream 事件流。还记得开头那个"我发出去的那一刻这一轮就锁死了、只能等它跑完"的本能假设吗？现在你不仅知道它是错的，还能说清它为什么错——agent 的心脏是一个会主动"回头看你有没有新话要说"的双层 while 循环，steering 就插在这个回头看的动作里。带着这个从"以为锁死"到"看懂可中断"的转变，我们停下来确认一下：你现在能做什么？

&emsp;&emsp;第一，你能读懂 `agent-loop.ts` 的双层 while 结构：外层 `while(true)` 维持心跳（第 216 行），内层 `while(hasMoreToolCalls || pendingMessages.length > 0)` 处理单轮内所有工具调用和插话（第 220 行）。第二，你能解释 steering 为什么是 agent 与 chatbot 的分水岭——它让 agent 成为"可中断、可引导的循环"，而不是一次性的请求—响应。第三，你能在 TUI 里认出事件流的关键节点：叫得出 `agent_start / turn_start / message_start / message_end / turn_end / agent_end` 这 6 个主骨架事件的名字和顺序，也知道 `AgentEvent` 全集是 10 种、分 4 组生命周期——工具执行组（`tool_execution_start/update/end`）和流式增量（`message_update`）补全了余下 4 种。

&emsp;&emsp;还有一个边界要点请你记住：你这一章用 Python MVP 重现的，是 **agent-core 这个内核模块的机制**。在完整的 OpenClaw 里，agent-core 是独立于 Gateway（网关）的——它只管"循环和事件"这个心跳逻辑，至于消息从哪个渠道进来、在哪台设备上执行，那是 Gateway 的事，不属于本节范围。把内核和外壳分清楚，你对 OpenClaw 的架构地图就有了第一块拼图。

&emsp;&emsp;心脏有了心跳，下一章我们看"心脏的手"——当模型在 loop 里决定要调用一个工具时，这次调用不是想调就能调的，它要穿过一道工具系统的关卡。我们会看清 plugin、capability、tool 三个概念各自的边界（谁是宿主、谁是分类标签、谁是被调用的工具），看清那道关卡到底有几步、凭什么放行或拦截，以及一个最容易讲反的设计：OpenClaw 为什么默认把执行权限开得很大。

---

## <center>第 3 章：工具系统 + 插件化外壳</center>

&emsp;&emsp;上一章我们看清了 agent 的心跳——它在 loop 里转，转到某一步模型说"我要调用 `exec` 去执行一条命令"。这一章我们就接着这个动作往下挖：这次工具调用是怎么被组织、被管控的。我们会先把 plugin、capability、tool 三个概念的边界分清楚（第 3.1 节）；然后看工具调用要穿过的那道关卡——它不是大家常以为的"几层权限"，而是确切的 8 步策略管线（第 3.2 节）；接着用 OpenClaw 自带的可观测面对账放行与拦截两种情形（第 3.3 节）；最后在第 3.4 节确认能力，并为后续的安全设计哲学埋一个伏笔。

&emsp;&emsp;这一章你需要建立的核心认知有三条：**第一，plugin / capability / tool 是三个不同维度的概念——plugin 是带版本契约的宿主扩展单元，capability 是 plugin 的功能分类标签，tool 是 plugin 注册暴露给 agent 调用的工具（tool 归属于 plugin）；第二，一次工具调用要按固定顺序穿过 8 个 policy step，每一步都可能收窄可用工具集；第三，OpenClaw 默认让 exec 工具"全开"，这不是漏洞，而是面向单一可信操作者的有意产品设计。** 第三条最容易讲反，我们会反复强调正确的框定。

### 3.1 三个概念的边界：plugin（宿主）· capability（分类）· tool（被调用）

&emsp;&emsp;我们先纠正一个很常见的混淆。很多人会把 tool 和 plugin 当成同一层的东西，或者把 capability 当成 tool 的别名，甚至以为这三个词构成一条"tool 包含于 plugin、plugin 又包含于 capability"的整齐套娃链。这个套娃理解是错的——我们待会用源码事实把它纠正过来。先记住一句话：**plugin 是宿主、capability 是分类标签、tool 是被调用的工具，三者各管一个维度，只有 tool 和 plugin 之间是真正的归属关系（tool ⊂ plugin）。**

&emsp;&emsp;先看 **plugin（插件）**。plugin 是 OpenClaw "万物皆插件"设计的核心单元——model provider、channel、工具能力都被统一成 plugin 来管理。每个 plugin 带一个 host 版本契约 `minHostVersion`，声明"我至少需要宿主到哪个版本"。这正是第 1 章我们举例看到的那行日志——`plugin requires OpenClaw >=2026.4.25, but this host is 2026.3.24; skipping load`——背后的机制：当 host 版本低于插件声明的 `minHostVersion` 要求时，插件就被跳过加载。这个契约对应真实源码 `src/plugins/manifest-registry.ts` 第 1026 行的 `checkMinHostVersion`——如果 host 太旧，OpenClaw 会 skip 掉这个 plugin 而不是崩溃。plugin 是其他两个概念的载体：它向 agent 注册 tool，同时声明自己属于哪些 capability 分类。

&emsp;&emsp;再看 **tool（工具）**。一个 tool 就是 agent 能调用的一个具体动作，比如 `exec`（执行命令）、`read`（读文件）、`write`（写文件）。在 OpenClaw 里，所有 tool 都遵循一个统一的契约类型 `AnyAgentTool`——不管是哪个工具，对外暴露的接口形状是一致的，这样 agent loop 才能用同一套逻辑调用它们。tool 是由 plugin 注册暴露出来的，所以 **tool ⊂ plugin**（一个 tool 必然归属于某个 plugin）——这是三个概念里唯一的包含关系。

&emsp;&emsp;把这个统一契约落到源码上看，`AnyAgentTool` 的主定义在 `src/agents/tools/common.ts`，它基于 agent-core 的 `AgentTool` 做了一层泛型擦除，方便 OpenClaw 把不同参数 schema 的工具统一放进 `AnyAgentTool[]` 管理。`AgentTool` 的运行时契约在 `packages/agent-core/src/types.ts`，而它继承的最基础模型工具契约 `Tool<TParameters>` 在 `packages/agent-core/src/llm.ts`：模型真正看见的核心三件套是 `name`（工具名）、`description`（告诉模型什么时候用）、`parameters`（TypeBox / JSON Schema 风格的参数 schema）。OpenClaw 又在这个基础上加了 `label`（TUI 展示名）、`execute`（真正执行函数）、`prepareArguments`（执行前修正模型参数）、`executionMode`（串行/并行偏好）和 `displaySummary`（UI 摘要）。以 `exec` 为例，工具对象在 `src/agents/agent-tools.ts` 里声明 `name: "exec"`、`label: "exec"`、动态 `description`、`parameters: execSchema` 和 `execute`；其中 `execSchema` 定义在 `src/agents/bash-tools.schemas.ts`，明确列出模型可传的 `command`、`workdir`、`env`、`timeout`、`pty`、`elevated`、`host` 等字段，`command` 是必填字符串，描述就是 `"Shell command to execute"`。所以模型不是随便拼参数调用 `exec`，而是按这个 schema 生成一次结构化工具调用。

&emsp;&emsp;最后看 **capability（能力分类）**。capability **不是**某个具体工具，也**不是**包含 plugin 的更外层容器，而是给 plugin 贴的一个**功能分类标签**——回答"这个 plugin 是哪一类 plugin"。OpenClaw 在 `src/plugins/inspect-shape.ts` 第 4 到 19 行定义了一个 `PluginCapabilityKind` 类型，确切地列举了 **15 种** 能力分类：`cli-backend`、`text-inference`、`embedding`、`speech`、`realtime-transcription`、`realtime-voice`、`media-understanding`、`transcript-source`、`image-generation`、`video-generation`、`music-generation`、`web-search`、`agent-harness`、`context-engine`、`channel`。**注意：`tool` 并不在这 15 种里**——所以"tool 是一种 capability"或"tool ⊂ plugin ⊂ capability"都是错的。capability 回答"这 plugin 归哪类"，tool 回答"这 plugin 能调用哪个动作"，两者是 plugin 的两个不同侧面。

&emsp;&emsp;把三者的关系正确串起来就是：**plugin 是带 `minHostVersion` 契约的宿主扩展单元；它注册若干 tool 供 agent 调用（tool ⊂ plugin）；同时它声明自己属于某些 capability 分类（15 种之一）。capability 不在 tool→plugin 的归属链上，它是描述 plugin 的分类维度。** 下面这张图把这个关系画出来，你照着它在脑子里过一遍。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102207137.png" width=60%></div>

&emsp;&emsp;现在你用 Python 把这三个概念搭出来跑一遍。运行后你会看到一条 "tool 归属 plugin、plugin 声明 capability 分类" 的关系打印，以及 `minHostVersion` 门控的两个结果——host 够新放行、host 过旧 skip。如果断言报错，说明概念关联或版本门控逻辑没搭对。

In [9]:
# Python 最小重现，非 OpenClaw 真实源码（真实实现见 TS: src/plugins/inspect-shape.ts、src/plugins/manifest-registry.ts:1026）
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

# ── capability 分类：plugin 的功能分类标签（不是 tool，也不是包含 plugin 的容器）──
# 对照真实源码 inspect-shape.ts:4-19 的 15 种，此处取教学子集；注意 tool 不在这 15 种里
class PluginCapabilityKind(str, Enum):
    AGENT_HARNESS  = "agent-harness"   # agent 运行时内核类 plugin
    CHANNEL        = "channel"         # 渠道接入类 plugin
    TEXT_INFERENCE = "text-inference"  # 文本推理（LLM）类 plugin
    WEB_SEARCH     = "web-search"      # 联网搜索类 plugin
    IMAGE_GEN      = "image-generation"  # 图像生成类 plugin

@dataclass
class AgentTool:
    """一个工具：plugin 注册暴露给 agent 调用的具体动作（tool 归属于 plugin，即 tool ⊂ plugin）。

    Args:
        name: 工具名，如 exec / read / write
        plugin: 反向引用，记录这个 tool 由哪个 plugin 注册提供（注册时回填）
    """
    name: str
    plugin: "Plugin" = None  # tool 必然挂在某个 plugin 下，注册时回填

@dataclass
class Plugin:
    """一个插件：带 host 版本契约的宿主扩展单元，统一管理 provider/channel/工具能力。

    plugin 是宿主载体——它声明自己属于哪些 capability 分类（贴标签），并注册若干 tool 供 agent 调用。
    capability 不是 plugin 的子集，而是描述 plugin 的功能分类标签。

    Args:
        name: 插件名
        min_host_version: semver floor，host 过旧则 skip 不崩溃
        capabilities: 该 plugin 声明的 capability 分类标签列表（15 种之一）
        tools: 该 plugin 注册暴露的具体 tool 列表
    """
    name: str
    min_host_version: str
    capabilities: list = field(default_factory=list)  # plugin 的分类标签，plugin 主动声明
    tools: list = field(default_factory=list)         # plugin 注册的工具，tool ⊂ plugin

    def declare_capability(self, kind: "PluginCapabilityKind"):
        # plugin 声明自己属于某个 capability 分类（贴标签，不是包含关系）
        if kind not in self.capabilities:
            self.capabilities.append(kind)

    def register_tool(self, tool: AgentTool):
        # 注册工具：回填 tool→plugin 反向引用，把该 tool 纳入本 plugin 的工具集
        tool.plugin = self          # 建立 tool→plugin 归属
        self.tools.append(tool)     # tool 进入 plugin 的工具列表

def host_version_ok(plugin: Plugin, host_version: str) -> bool:
    """简化版 minHostVersion 门控：host 版本 >= 插件要求即放行。

    Args:
        plugin: 待检查的插件
        host_version: 当前宿主版本号字符串，如 "1.5.0"
    Returns:
        bool: True=放行加载，False=host 过旧应 skip（真实见 manifest-registry.ts:1026）
    """
    # 把 "1.5.0" 解析成 (1,5,0) 做元组比较，简化了完整 semver 规则
    h    = tuple(int(x) for x in host_version.split("."))
    need = tuple(int(x) for x in plugin.min_host_version.split("."))
    return h >= need

# 构造一个 builtin-shell 插件：它是宿主扩展单元，声明自己属于 agent-harness 分类，并注册 exec 工具
core_plugin = Plugin(name="builtin-shell", min_host_version="1.2.0")
core_plugin.declare_capability(PluginCapabilityKind.AGENT_HARNESS)  # plugin 给自己贴分类标签
exec_tool = AgentTool(name="exec")                                  # tool 本身不带 capability
core_plugin.register_tool(exec_tool)                                # plugin 注册 tool，建立 tool ⊂ plugin

print(">>> Cell F 三个概念的边界：")
# 唯一的归属关系：tool ⊂ plugin（exec 反查得到它属于 builtin-shell）
print(f"  tool '{exec_tool.name}' 归属于 plugin '{exec_tool.plugin.name}'（tool ⊂ plugin）")
# plugin 声明的 capability 是"分类标签"，不是包含 tool、也不是包含 plugin 的容器
print(f"  plugin '{core_plugin.name}' 声明的 capability 分类标签: {[c.value for c in core_plugin.capabilities]}")
print(f"  plugin '{core_plugin.name}' 注册的 tool: {[t.name for t in core_plugin.tools]}")

# ── Tier 1 断言：capability 枚举存在 / tool 反查 plugin / plugin 持有分类标签 / minHostVersion 门控 ──
assert PluginCapabilityKind.AGENT_HARNESS.value == "agent-harness", "capability 分类枚举必须存在"
assert exec_tool.plugin is core_plugin, "tool 必须能反查到所属 plugin（tool ⊂ plugin）"
assert PluginCapabilityKind.AGENT_HARNESS in core_plugin.capabilities, "plugin 必须持有它声明的 capability 分类标签"
assert exec_tool in core_plugin.tools, "plugin 的工具集必须含已注册的 tool"
assert host_version_ok(core_plugin, "1.5.0") is True,  "host 够新应放行"
assert host_version_ok(core_plugin, "1.0.0") is False, "host 过旧应 skip"
print("[OK] Cell F 断言通过：capability 分类存在 / tool→plugin 归属 / plugin 持分类标签 / minHostVersion 门控正确")

>>> Cell F 三个概念的边界：
  tool 'exec' 归属于 plugin 'builtin-shell'（tool ⊂ plugin）
  plugin 'builtin-shell' 声明的 capability 分类标签: ['agent-harness']
  plugin 'builtin-shell' 注册的 tool: ['exec']
[OK] Cell F 断言通过：capability 分类存在 / tool→plugin 归属 / plugin 持分类标签 / minHostVersion 门控正确


&emsp;&emsp;这段代码把三个概念的边界落到了可运行的对象上：`exec` 这个 tool 反查得到它归属于 `builtin-shell` 这个 plugin（tool ⊂ plugin），这个 plugin 主动声明自己属于 `agent-harness` 这个 capability 分类（贴标签，不是被 capability 包含），而 `minHostVersion` 门控保证了 host 太旧时插件被安全 skip 而非崩溃。

> **【常见误区】**：把 tool、plugin、capability 当成可以互换的词，或者串成"tool ⊂ plugin ⊂ capability"的整齐套娃。后果是读策略管线时会以为"过滤 tool"和"过滤 plugin"是一回事，或者以为 capability 是包住 plugin 的更外层容器。正确边界：tool 是 plugin 注册暴露的工具（tool ⊂ plugin，这是唯一的包含关系）；capability 是贴在 plugin 上的功能分类标签（不是 tool 的别名，也不包含 plugin，`tool` 根本不在 15 种 capability 里）。排查方法：问自己"它能不能独立存在"——tool 必须挂在某个 plugin 下；capability 只是描述 plugin 是哪一类的标签。

> **【Tier 1 验证已通过】**：上面的断言独立验证了三个概念的四个共性——capability 分类枚举存在、tool 能反查所属 plugin、plugin 持有它声明的分类标签、minHostVersion 门控正确放行/拦截。下一节进入工具调用真正的关卡：8 步策略管线。

### 3.2 8 步工具策略管线——放行与拦截

&emsp;&emsp;现在到了这一章信息量最大的一节。当模型在 loop 里决定调用 `exec` 时，这次调用不是直接就执行的——它要穿过一道**策略管线**。这道管线常常被旧的教学材料笼统地讲成"几层权限门"，但我们打开真实源码数清楚，它是**确切的 8 个 policy step**，对应 `src/agents/tool-policy-pipeline.ts` 里 `buildDefaultToolPolicyPipelineSteps` 函数返回的 8 个 step。下面把这 8 步按源码里的 label 顺序列出来，你逐行核对。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>工具策略管线的 8 个 policy step（按源码顺序）</font></p>
<div class="center">

| 步骤 | 源码 label | 这一步在过滤什么 |
|------|-----------|------------------|
| 1 | `tools.profile (${profile})` | Profile 层过滤（如 full / coding / minimal）|
| 2 | `tools.byProvider.profile (${providerProfile})` | Provider 维度的 profile 过滤（有 provider 时拼上名字，无则退化为静态形式）|
| 3 | `tools.allow` | 全局 allow 策略 |
| 4 | `tools.byProvider.allow` | Provider 维度的全局 allow |
| 5 | `agents.${agentId}.tools.allow` | per-agent 的 allow 策略 |
| 6 | `agents.${agentId}.tools.byProvider.allow` | per-agent + provider 双维度 |
| 7 | `group tools.allow` | group 层 allow |
| 8 | `tools.toolsBySender` | sender（消息来源）维度过滤 |

</div>

#### 8 个 step 到底各管哪一层权限？

&emsp;&emsp;先给一个总心智模型：这 8 步不是“任意一步允许就能执行”，而是**逐层取交集**。工具集一开始可能很宽，经过 profile、provider、全局配置、agent、group、sender 这些维度后，只会越来越窄，不会在后面的步骤里重新变宽。也就是说，`exec` 想真正执行，必须在 8 步里每一步都没有被过滤掉。下面的 JSON 都是教学示意，重点是理解“这一层按什么维度收窄工具”，不是要求你现在背完整配置格式。

&emsp;&emsp;这些规则主要来自 OpenClaw 主配置文件（默认是 `~/.openclaw/openclaw.json`，也可以被 `OPENCLAW_CONFIG_PATH` 覆盖），但不要把它理解成“8 个 step 对应 8 个独立配置文件”。更准确地说：OpenClaw 会从同一份 runtime config 里读出全局、agent、provider、channel group、sender 等配置，再结合当前 agent、provider、session、sender 上下文，解析成下面这 8 个 policy step。

| 步骤 | policy 来源 | 主要配置路径 |
|------|-------------|--------------|
| 1. `tools.profile` | profile 名字来自配置，profile 内容是内置预设 | `tools.profile` 或 `agents.list[].tools.profile` |
| 2. `tools.byProvider.profile` | provider 维度的 profile | `tools.byProvider.<provider>.profile` 或 `agents.list[].tools.byProvider.<provider>.profile` |
| 3. `tools.allow` | 全局工具策略 | `tools.allow` / `tools.alsoAllow` / `tools.deny` |
| 4. `tools.byProvider.allow` | 全局 provider 工具策略 | `tools.byProvider.<provider>.allow` / `tools.byProvider.<provider>.deny` |
| 5. `agents.${agentId}.tools.allow` | 单个 agent 工具策略 | `agents.list[].tools.allow` / `agents.list[].tools.deny` |
| 6. `agents.${agentId}.tools.byProvider.allow` | 单个 agent + provider 工具策略 | `agents.list[].tools.byProvider.<provider>.allow` / `deny` |
| 7. `group tools.allow` | channel group / room / channel 工具策略 | `channels.<channel>.groups.<groupId>.tools`，不同 channel 结构略有不同 |
| 8. `tools.toolsBySender` | 全局或 agent 级 sender 工具策略 | `tools.toolsBySender` 或 `agents.list[].tools.toolsBySender` |

&emsp;&emsp;**第 1 步 `tools.profile (${profile})`** 管的是当前运行模式的基础工具集。`profile` 可以理解成“这次 agent 运行的大权限档位”，比如 `full` 更宽，`minimal` 更窄，`coding` 偏开发场景。它通常是第一道粗过滤：如果 profile 本身不允许 `exec`，后面就不用看了，`exec` 会直接被挡在入口处。

```text
profile = full     -> 初始工具集很宽，exec 可以继续往第 2 步走
profile = minimal  -> 初始工具集很窄，exec 可能在第 1 步就被挡住
```

&emsp;&emsp;**第 2 步 `tools.byProvider.profile (${providerProfile})`** 管的是 provider 维度下的 profile 过滤。不同模型 provider 或后端的信任边界可能不同：本地 provider 可以更宽，远程 provider 可能只允许读文件或检索。它回答的问题是：在当前 provider 之下，这个 profile 还能保留哪些工具？

```text
当前 profile = full
provider = local   -> 仍按 full 理解，exec 继续往后走
provider = remote  -> providerProfile 收紧，只保留 read / web_search，exec 在第 2 步被挡住
```

&emsp;&emsp;**第 3 步 `tools.allow`** 是全局 allow 策略，可以理解成系统层面的“总闸”。即使 profile 是 `full`，如果全局 allow 只写了 `read`、`write`，那 `exec` 也会在这一层被过滤掉。它适合表达“整个 OpenClaw 实例最多只能用这些工具”。

```json
{
  "tools": {
    "allow": ["read", "write"]
  }
}
```

&emsp;&emsp;这个例子的意思是：全局最多只允许 `read` 和 `write`。所以哪怕第 1 步 `profile=full` 放过了 `exec`，`exec` 也会在第 3 步被全局总闸收掉。

&emsp;&emsp;**第 4 步 `tools.byProvider.allow`** 是 provider 维度的全局 allow。它比第 3 步更细：不是所有 provider 共用一套总闸，而是每个 provider 可以有自己的总闸。比如 local provider 允许 `exec`，remote provider 只允许 `read`，同一个 agent 切换 provider 后，工具权限也会跟着变化。

```json
{
  "tools": {
    "byProvider": {
      "local": {
        "allow": ["*"]
      },
      "remote": {
        "allow": ["read", "web_search"]
      }
    }
  }
}
```

&emsp;&emsp;这个例子的意思是：所有 agent 只要走 `local` provider，工具总闸比较宽；只要走 `remote` provider，全局上就只剩 `read` 和 `web_search`，`exec` 会在第 4 步被挡住。

&emsp;&emsp;**第 5 步 `agents.${agentId}.tools.allow`** 管的是单个 agent 的工具权限。OpenClaw 里可能有 main agent、reviewer agent、planner agent 等不同 agent，它们不应该天然拥有同样的工具集。main agent 可以允许 `exec`，reviewer agent 可能只允许 `read`，这一步就是按 agent 身份收窄工具。

```json
{
  "agents": {
    "main": {
      "tools": {
        "allow": ["read", "write", "exec"]
      }
    },
    "reviewer": {
      "tools": {
        "allow": ["read"]
      }
    }
  }
}
```

&emsp;&emsp;这个例子的意思是：`main` agent 可以继续使用 `exec`，但 `reviewer` agent 只能读，不能执行命令。即使 reviewer 前面几步都没被挡，到了第 5 步也会失去 `exec`。

&emsp;&emsp;**第 6 步 `agents.${agentId}.tools.byProvider.allow`** 管的是“单个 agent + provider”这个组合权限。它比第 5 步还细，同时考虑两个问题：当前是哪一个 agent，以及当前用的是哪一个 provider。比如同一个 `main` agent，在本地 provider 下可以执行命令，在远程 provider 下只能读文件。这一步用来表达更精确的信任边界，避免把 agent 权限和 provider 权限混成一锅。

```json
{
  "agents": {
    "main": {
      "tools": {
        "byProvider": {
          "local": {
            "allow": ["*"]
          },
          "remote": {
            "allow": ["read"]
          }
        }
      }
    }
  }
}
```

&emsp;&emsp;这个例子的含义是：`main` agent 本身并不是永远能用 `exec`，它要看自己当前接在哪个 provider 上。走 `local` provider 时，`allow=["*"]`，`exec` 可以继续往后走；走 `remote` provider 时，只留下 `read`，`exec` 会在第 6 步被过滤掉。

&emsp;&emsp;**第 7 步 `group tools.allow`** 管的是 group 层权限。多个 agent 可以归到同一个 group，比如 dev group、readonly group、automation group。group allow 的作用是给一组 agent 统一套一层工具上限：只要 group 没放行 `exec`，即使某个 agent 自己的 allow 里有 `exec`，最终也会被 group 层收掉。

```json
{
  "groups": {
    "readonly": {
      "tools": {
        "allow": ["read"]
      }
    },
    "dev": {
      "tools": {
        "allow": ["read", "write", "exec"]
      }
    }
  }
}
```

&emsp;&emsp;这个例子的意思是：如果某个 agent 被放进 `readonly` group，那么它在第 5 步就算允许 `exec`，第 7 步也会被 group 上限压回只读。group 层适合给一批 agent 统一兜底。

&emsp;&emsp;**第 8 步 `tools.toolsBySender`** 管的是 sender，也就是消息来源。来自本地 TUI 的请求、来自外部聊天渠道的请求、来自自动化流程的请求，信任程度并不一样。同一个 agent、同一个 provider，在本地 TUI 里可以允许 `exec`，但外部群聊触发时可能只允许 `read` 或 `web_search`。这一层把“谁发起的请求”也纳入工具权限判断。

```json
{
  "tools": {
    "toolsBySender": {
      "tui": ["read", "write", "exec"],
      "external-chat": ["read", "web_search"]
    }
  }
}
```

&emsp;&emsp;这个例子的意思是：同一句“帮我列一下目录”，如果是你在本地 TUI 里发起，`exec` 可以继续；如果是外部聊天渠道触发，`exec` 会在 sender 层被挡住。第 8 步解决的是“入口来源是否可信”的问题。

&emsp;&emsp;所以最终权限可以这样理解：`profile` 先给基础工具集，provider、全局、agent、group、sender 再一层层收窄。`profile=full` 只是说明第 1 步给得很宽，并不等于后面 7 步自动通过。举个例子：`profile=full` 放行了 `exec`，`tools.allow=["*"]` 也放行了 `exec`，`agents.main.tools.allow` 也有 `exec`，但如果 `group tools.allow=["read", "write"]`，那 `exec` 仍然会被第 7 步拦截。**最终结果看的是 8 步共同留下来的交集，而不是某一层单独说了算。**

&emsp;&emsp;这道管线的工作方式，对应源码里的 `applyToolPolicyPipeline`——它逐个 step 应用过滤，每一步都可能收窄当前可用的工具集（strip 掉一些工具，或调整 plugin 的 allowlist）。一个工具只有穿过全部 8 步都没被任何一步过滤掉，才最终可用。这里有一个理解上的关键点要先说清。

> **【常见误区】**：把这道管线理解成"会弹出 UI 让你点同意"的审批流程。后果是你会去 OpenClaw 界面找审批弹窗，找不到就以为机制没生效。实际上这是**纯过滤**（allowlist 收窄），不弹任何 UI；当一个工具被拦截时，它表现为日志里出现一行 `tools: <工具名> blocked by before_tool_call: <reason>`——这行文案来自真实源码 `src/agents/agent-tool-definition-adapter.ts` 第 371 行。排查方法：用 `--log-level debug` 跑，被拦的工具会在日志里留下这行 blocked 记录。

&emsp;&emsp;讲到工具策略，就绕不开 OpenClaw 一个最容易被讲反的设计——exec 工具的默认值。我们直接看源码事实：`src/infra/exec-approvals.ts` 第 205 行定义 `DEFAULT_SECURITY = "full"`，第 206 行定义 `DEFAULT_ASK = "off"`。翻译过来就是：默认情况下，主 agent 在 gateway/node 上执行命令是**全开、不弹审批提示**的。

&emsp;&emsp;看到这里，几乎所有人的第一反应都是："这不就是个安全漏洞吗？默认让 AI 随便执行命令？" **这个反应是错的，而且是这整节最重要的一个校准点。** 我们必须把它框定正确。

> **【踩坑预警 · 最重要的校准】**：千万不要把 exec 默认 `full`/`off` 讲成"OpenClaw 默认不安全"或"这是安全漏洞"。这是 OpenClaw 官方文档里明确声明的**有意设计**——`docs/gateway/security/index.md` 第 126 行原文写道：默认让单一可信操作者（trusted single-operator）的 host exec 不弹审批，"That default is intentional UX, not a vulnerability by itself"（这个默认是有意的产品体验，本身不是漏洞）。正确的说法是：**对单一可信操作者的产品设计来说，exec approvals 是 operator intent（操作者意图）的 guardrails（护栏），而不是 hostile multi-tenant（敌对多租户）的隔离。** 如果你的场景是多租户或对抗性环境，需要自行加固 sandbox 和 host 隔离——官方文档同一处也明确这么说了。

&emsp;&emsp;为什么这个框定比"默认不安全"更有价值？因为它教给你的是一个**信任边界的产品决策思维**：一个工具的默认值开多大，取决于它面对的是谁。OpenClaw 假设它服务的是"信任自己机器的那一个操作者"，所以默认敞开以换取流畅体验，同时用策略管线和 approvals 做兜底护栏。这是工程上的取舍，不是疏忽。

&emsp;&emsp;现在你用 Python 把 8 步管线重现出来，跑两个案例：一个放行、一个拦截。这里先说清一个容易让人对不上号的编号差异：上面那张表为了符合人类阅读习惯，把 8 步编号成 **1 到 8**；而下面代码里用的是 Python 列表下标，是 **0 到 7**（第 0 个元素就是表格里的步骤 1，即 Profile 层）。所以你运行后会看到拦截案例报"在第 0 步被拦截"——这个"第 0 步"就是表格里的步骤 1（Profile 层），不要在表格里去找一个不存在的"步骤 0"。放行案例则会穿过全部 8 步。如果放行案例没穿满 8 步、或拦截案例没在预期步停下，断言会报错。

In [10]:
# Python 最小重现，非 OpenClaw 真实源码（真实实现见 TS: src/agents/tool-policy-pipeline.ts、src/infra/exec-approvals.ts:205,206）
# ── Cell G：8 步工具策略管线（放行 vs 拦截）──
from dataclasses import dataclass
from typing import Optional  # 本 cell 独立可运行所需（dataclass 字段用到 Optional）
@dataclass
class PolicyStep:
    """一个策略步：对照源码 buildDefaultToolPolicyPipelineSteps 返回的 8 个 step 之一。

    Args:
        label: 步骤名，与源码 8 个 label 对齐
        allow: 该步 allowlist；None=该层未配置 policy，不收窄透传；含 "*"=全开，否则需命中名单
    """
    label: str
    allow: Optional[list] = None

    def permits(self, tool_name: str) -> bool:
        # 该步是否放行某工具：None 表示该层未配置，不收窄；含 "*" 全开；否则要求工具名命中 allowlist
        if self.allow is None:
            return True
        if "*" in self.allow:
            return True
        return tool_name in self.allow

def build_default_pipeline(agent_id="main", profile="full"):
    # 构造 8 步默认管线，label 与源码 tool-policy-pipeline.ts 一一对应
    # 注意：full 只代表第 1 步 profile 很宽；后 7 步 allow=None 表示这些层未配置额外收窄。
    # minimal profile 仅放行 session_status（呼应源码 profile 定义），所以 exec 会在第 1 步被拦。
    return [
        PolicyStep(f"tools.profile ({profile})", allow=["*"] if profile == "full" else ["session_status"]),
        PolicyStep("tools.byProvider.profile", allow=None),
        PolicyStep("tools.allow", allow=None),
        PolicyStep("tools.byProvider.allow", allow=None),
        PolicyStep(f"agents.{agent_id}.tools.allow", allow=None),
        PolicyStep(f"agents.{agent_id}.tools.byProvider.allow", allow=None),
        PolicyStep("group tools.allow", allow=None),
        PolicyStep("tools.toolsBySender", allow=None),
    ]

def apply_tool_policy_pipeline(tool_name: str, steps: list):
    """逐步过滤工具：任一步不放行即拦截。

    Args:
        tool_name: 待检查的工具名
        steps: 8 步策略管线
    Returns:
        tuple(放行bool, 穿过/拦截的步index, 拦截步label或None)
    """
    # 按顺序走每一步，第一个不放行的步就拦截并报告位置
    for idx, step in enumerate(steps):
        if not step.permits(tool_name):
            return False, idx, step.label
    return True, len(steps), None

# 案例 1：profile=full + 后 7 步没有额外限制，exec 才会全程放行
steps_full = build_default_pipeline(profile="full")
ok1, passed1, _ = apply_tool_policy_pipeline("exec", steps_full)
print(">>> Cell G 案例 1（profile=full，后 7 步未额外收紧，调用 exec）：")
print(f"  放行 = {ok1}，穿过 {passed1}/8 步")

# 案例 2：操作者把 profile 收紧为 minimal，exec 在第 0 步（即表格步骤 1，Profile 层）被拦
steps_min = build_default_pipeline(profile="minimal")
ok2, stopped2, blk2 = apply_tool_policy_pipeline("exec", steps_min)
print(">>> Cell G 案例 2（minimal profile，调用 exec）：")
print(f"  放行 = {ok2}，在第 {stopped2} 步被拦截，拦截步 = '{blk2}'")
print(f"  （真实拦截在日志体现为：tools: exec blocked by before_tool_call: <reason>）")

# 案例 3：即使 profile=full，后续任一层收紧也能拦截 exec；这里模拟 group 层只允许 read/write
steps_full_group_restricted = build_default_pipeline(profile="full")
steps_full_group_restricted[6] = PolicyStep("group tools.allow", allow=["read", "write"])
ok3, stopped3, blk3 = apply_tool_policy_pipeline("exec", steps_full_group_restricted)
print(">>> Cell G 案例 3（profile=full，但 group tools.allow 收紧，调用 exec）：")
print(f"  放行 = {ok3}，在第 {stopped3} 步被拦截，拦截步 = '{blk3}'")

# ── Tier 1 断言 ──
assert ok1 is True and passed1 == 8,   "profile=full 且后 7 步无额外限制时，exec 应穿过全部 8 步"
assert ok2 is False and stopped2 == 0, "minimal profile 下 exec 必须在第 0 步（profile 层）被拦"
assert ok3 is False and stopped3 == 6, "profile=full 但 group 层收紧时，exec 必须在第 6 步被拦"
assert blk2 is not None,                "拦截原因（step label）必须非空"
assert blk3 == "group tools.allow",     "案例 3 的拦截步必须是 group tools.allow"
assert len(steps_full) == 8,            "默认管线必须恰好是 8 步"
print("[OK] Cell G 断言通过：默认宽松放行 / profile 收紧拦截 / group 收紧拦截 / 管线 8 步")

>>> Cell G 案例 1（profile=full，后 7 步未额外收紧，调用 exec）：
  放行 = True，穿过 8/8 步
>>> Cell G 案例 2（minimal profile，调用 exec）：
  放行 = False，在第 0 步被拦截，拦截步 = 'tools.profile (minimal)'
  （真实拦截在日志体现为：tools: exec blocked by before_tool_call: <reason>）
>>> Cell G 案例 3（profile=full，但 group tools.allow 收紧，调用 exec）：
  放行 = False，在第 6 步被拦截，拦截步 = 'group tools.allow'
[OK] Cell G 断言通过：默认宽松放行 / profile 收紧拦截 / group 收紧拦截 / 管线 8 步


&emsp;&emsp;这段代码完成的是放行与拦截的对比演示。案例 1 里 `profile=full` 只让第一步很宽，后 7 步因为没有额外 policy 才一路透传，所以 `exec` 穿过 8 步；案例 2 里操作者主动把 profile 收紧成 `minimal`，`exec` 在第一步（profile 层）就被挡下；案例 3 则刻意保留 `profile=full`，但把 group 层收紧到只允许 `read/write`，于是 `exec` 会在第 7 个 policy step（Python 下标第 6 步）被挡下。注意这里的拦截不是"漏洞导致的意外",而是**操作者主动收紧策略后护栏生效**的结果——这正呼应了上面那个框定：默认敞开，但管线随时能收。下一节我们用 OpenClaw 自带的可观测面把这些情形对一次账。

> **【Tier 1 验证已通过】**：上面的断言独立验证了管线的四个行为——默认宽松案例穿过全部 8 步、profile 收紧案例在第 1 个 policy step 停下、`profile=full` 但 group 收紧案例仍会被后续步骤拦截、管线恰好 8 步。

### 3.3 章末收口

&emsp;&emsp;这一章我们从"模型想调用一个工具"这个动作出发，挖清了工具系统的组织方式和管控机制。现在确认一下你的收获——下面这三件事你应该都能做到了。

&emsp;&emsp;第一，你能分清 plugin / capability / tool 三个概念的边界——plugin 是带 `minHostVersion` 契约的宿主扩展单元，它注册 tool 供 agent 调用（tool ⊂ plugin），并声明自己属于某个 capability 分类（15 种之一，`tool` 不在其中）；记住 capability 是贴在 plugin 上的分类标签，不是包住 plugin 的容器。第二，你能说出 8 步策略管线每一步在过滤什么，从 `tools.profile` 一直到 `tools.toolsBySender`，并且知道它是纯过滤而非 UI 审批。第三，也是最重要的——你能正确解释为什么 exec 默认 `full`/`off` 不是漏洞：它是面向单一可信操作者的有意产品 UX，approvals 是护栏而非多租户隔离。

&emsp;&emsp;给你一个更具体的自测：如果你现在能不看课件、用自己的话说出这句——"exec 默认 `full`/`off` 是面向单一可信操作者的产品 UX，不是漏洞；`profile=full` 只是让第 1 步很宽，不等于跳过后面 7 步；真要拦截，既可以把 profile 从 `full` 改成 `minimal`，也可以在 `tools.allow`、agent、group、sender 等后续层继续收紧"——那这一章最核心的认知校正你就真的拿到了。

&emsp;&emsp;Cell G 已经把这个认知用三条路验证过了：默认 `full` 且后续层没有额外收紧时，exec 穿过全部 8 步被放行；操作者把 profile 收紧后，exec 会在 profile 层被管线拦截；即使 profile 仍是 `full`，只要 group 层继续收紧，exec 也会在后续步骤被拦截。三路行为合在一起，正好诠释了"默认敞开、护栏可收"的信任边界设计。需要再次强调——**这段 Python 是机制本质的最小模型，OpenClaw 真实的 exec-approvals 远比它复杂**（有 allowlist、ask 模式、请求上下文绑定等），我们只保留了"策略管线决定放行/拦截"这一条主干。

&emsp;&emsp;我们在这里把这个"信任 vs 约束"的张力点明，因为它是后续安全设计章节的核心伏笔。OpenClaw 的设计哲学里有一个反复出现的张力：**给操作者足够的信任以换取流畅（默认敞开），同时用分层的策略和护栏兜底（管线随时能收）。** 我们这一章看到的 exec 默认值，就是这个张力最具体的一个落点。后续讲安全纵深时，你会看到 deny / allowlist / full 三档加上 sandbox 隔离构成的完整图景，而那一切的起点，就是今天这个"默认 full 不是漏洞"的正确认知。

&emsp;&emsp;到这里，你已经从"agent 的心跳"一路看到了"心脏的手如何受控"。但我们还欠一个交代——第 3 章只展开了 exec 的 `full` 这一档默认值，而它其实还有另外两档安全级别没讲。下一章我们就沿着今天埋下的"信任 vs 约束"这条线，把 OpenClaw 完整的安全纵深图谱补齐，并在最后把贯穿整套设计的张力哲学收口。今天这把"默认 full 不是漏洞"的认知钥匙，正是打开下一章的起点。

---

## <center>第 4 章：安全纵深 + 设计哲学</center>

&emsp;&emsp;第 3 章我们把一个最容易讲反的设计校准对了——exec 默认 `full`/`off` 不是漏洞，而是面向单一可信操作者的有意产品 UX。这一章我们站在那个已经建立好的认知上往前走一步：既然 `full` 是合理的默认，那另外两档（`deny` 和 `allowlist`）长什么样、什么场景下该用？这是第 4.1 节要补全的图谱。然后我们看安全体系的另一层纵深——当你真的需要把 agent 关进隔离环境时，OpenClaw 的 Docker sandbox 提供了哪些手段（第 4.2 节）。最后，我们把整套设计背后的"信任 vs 约束"张力点透，并对这一节课的五章做一次能力回顾收口（第 4.3 节）。

&emsp;&emsp;这一章是整节课的哲学收束，代码很少、概念为主。需要先说清楚一件事：**我们不会再重复第 3 章讲过的"full 是不是漏洞"那个话题**——那个认知你已经拿到了。这一章只做两件新事：一是把安全档位补全（从只知道 `full`，到知道完整的三档三态），二是把设计哲学拔高（从单个 exec 默认值，到贯穿系统的张力思维）。我们从补全 exec 的安全档位开始。

### 4.1 exec 三档完整图谱：deny / allowlist / full

&emsp;&emsp;先把第 3 章和本节之间的关系校准一下：`full/off` 是 exec 的默认安全档位和审批态；在后续 policy 层没有额外收紧时，exec 会顺利穿过 8 步。但如果 group、agent、provider、sender 等后续层收紧，即使 `profile=full`，exec 仍然会被拦。第 3 章我们看到 exec 的默认安全级别是 `full`，对应源码 `src/infra/exec-approvals.ts` 第 24 行的类型定义。现在我们把那行定义完整看一遍——它其实是一个三选一的联合类型：`ExecSecurity = "deny" | "allowlist" | "full"`。也就是说，`full` 只是三档里最宽松的那一档，另外两档我们之前没展开。同样地，第 25 行还定义了一个控制"是否弹审批"的三态类型：`ExecAsk = "off" | "on-miss" | "always"`，第 3 章我们只见过默认的 `off`。这一节我们把这"三档 × 三态"补全。

> **【关于同一文件的两组行号】**：你可能注意到 `exec-approvals.ts` 在本课出现了两组行号，它们指的是不同东西，别搞混——第 24 / 25 行是**类型定义**，回答"安全档位/审批态各有哪几种可选值"（`deny`/`allowlist`/`full` 和 `off`/`on-miss`/`always`）；而第 3 章见过的第 205 / 206 行是**默认值常量**，回答"在这些可选值里实际默认选了哪个"（结果是 `full` 和 `off`）。一句话：:24/:25 列出"有哪些选项"，:205/:206 指定"默认选了哪个"。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>exec 安全的两个维度</font></p>
<div class="center">

| 维度 | 源码类型 | 可选值 | 这一维管什么 | 含义与适用场景 |
|------|----------|--------|--------------|----------------|
| 安全档位 | `ExecSecurity` | `deny` | 能不能执行 | 禁止所有 exec，最严格，适合不信任环境、多租户环境 |
| 安全档位 | `ExecSecurity` | `allowlist` | 能不能执行 | 只允许白名单里的命令，适合受控生产环境 |
| 安全档位 | `ExecSecurity` | `full` | 能不能执行 | 全部允许，默认档，适合本机 trusted-operator 场景 |
| 审批触发 | `ExecAsk` | `off` | 要不要审批 | 不弹审批，默认值 |
| 审批触发 | `ExecAsk` | `on-miss` | 要不要审批 | 白名单没命中时才询问，通常配合 `allowlist` 使用 |
| 审批触发 | `ExecAsk` | `always` | 要不要审批 | 每次 exec 都询问，最谨慎但交互成本最高 |

</div>

&emsp;&emsp;先看 **ExecSecurity 的三档**，从严到宽排列。最严的是 `"deny"`——禁止一切 exec 调用，agent 完全不能在这个环境里执行 shell 命令；它适用于多租户或你完全不信任的环境。中间是 `"allowlist"`——只放行你在白名单里明确声明过的命令模式，没在名单上的一律拒绝；它适用于生产受控、需要精细管控的 operator 场景。最宽的是 `"full"`——全放行，也就是第 3 章讲透的那个默认值，适用于单一可信操作者（trusted-operator）在自己机器上使用的场景。

&emsp;&emsp;再看 **ExecAsk 的三态**，它控制的是"执行前要不要弹一个审批提示让你确认"。`"off"` 是静默执行、不弹任何审批（第 3 章见过的默认值）；`"on-miss"` 是只在 allowlist 未命中时才弹审批——它天然要配合 `allowlist` 档位使用；`"always"` 是每一次 exec 都弹审批，每条命令都要弹一次确认，执行越频繁确认越多。把这两个维度交叉起来，就得到一张"三档 × 三态"的组合矩阵——下面我们用 Python 把这两个枚举搭出来，跑一遍典型组合的语义，让你对它们的搭配有个具体的手感。运行后你会看到三种典型组合各自的语义被打印出来，并且断言会确认三档/三态枚举完整、默认值确实是 `full`/`off`。

In [11]:
# Python 最小重现，非 OpenClaw 真实源码（真实实现见 TS: src/infra/exec-approvals.ts:24,25,205,206）
from enum import Enum

# ── exec 安全档位三档：对照源码 exec-approvals.ts:24 ExecSecurity（从严到宽）──
class ExecSecurity(str, Enum):
    DENY      = "deny"       # 禁止一切 exec：多租户/不信任环境
    ALLOWLIST = "allowlist"  # 仅放行白名单命中的命令：受控生产
    FULL      = "full"       # 全放行（默认）：trusted-operator 单人场景

# ── exec 审批触发三态：对照源码 exec-approvals.ts:25 ExecAsk ──
class ExecAsk(str, Enum):
    OFF     = "off"      # 静默执行不弹审批（默认）
    ON_MISS = "on-miss"  # allowlist 未命中才弹审批（配合 allowlist 用）
    ALWAYS  = "always"   # 每次 exec 都弹审批，最谨慎

# 源码默认值：exec-approvals.ts:205,206
DEFAULT_SECURITY = ExecSecurity.FULL  # DEFAULT_SECURITY = "full"
DEFAULT_ASK      = ExecAsk.OFF        # DEFAULT_ASK = "off"

def describe_combo(sec: ExecSecurity, ask: ExecAsk) -> str:
    """给出某个 security × ask 组合的一句话语义。

    Args:
        sec: 安全档位（deny/allowlist/full 之一）
        ask: 审批触发态（off/on-miss/always 之一）
    Returns:
        str: 该组合的适用场景描述
    """
    # 三种典型组合的语义映射；其余组合给通用描述
    table = {
        (ExecSecurity.FULL,      ExecAsk.OFF):     "默认组合：全放行 + 静默，trusted-operator 流畅体验",
        (ExecSecurity.ALLOWLIST, ExecAsk.ON_MISS): "受控生产：白名单放行，未命中时弹审批兜底",
        (ExecSecurity.DENY,      ExecAsk.OFF):     "最严锁定：禁止一切 exec，多租户/不信任环境",
    }
    return table.get((sec, ask), f"{sec.value}+{ask.value}：自定义组合，按需搭配")

print(">>> Cell N1 exec 三档 × 三态典型组合：")
print(f"  [默认] {DEFAULT_SECURITY.value}/{DEFAULT_ASK.value} → {describe_combo(DEFAULT_SECURITY, DEFAULT_ASK)}")
for sec, ask in [(ExecSecurity.ALLOWLIST, ExecAsk.ON_MISS), (ExecSecurity.DENY, ExecAsk.OFF)]:
    print(f"  {sec.value}/{ask.value} → {describe_combo(sec, ask)}")

# ── Tier 1 断言：三档/三态枚举完整，默认值与源码一致 ──
assert {e.value for e in ExecSecurity} == {"deny", "allowlist", "full"}, "ExecSecurity 必须恰好覆盖三档"
assert {e.value for e in ExecAsk} == {"off", "on-miss", "always"}, "ExecAsk 必须恰好覆盖三态"
assert DEFAULT_SECURITY == ExecSecurity.FULL and DEFAULT_ASK == ExecAsk.OFF, "默认必须是 full/off"
print("[OK] Cell N1 断言通过：三档/三态枚举完整 + 默认 full/off 与源码一致")

>>> Cell N1 exec 三档 × 三态典型组合：
  [默认] full/off → 默认组合：全放行 + 静默，trusted-operator 流畅体验
  allowlist/on-miss → 受控生产：白名单放行，未命中时弹审批兜底
  deny/off → 最严锁定：禁止一切 exec，多租户/不信任环境
[OK] Cell N1 断言通过：三档/三态枚举完整 + 默认 full/off 与源码一致


&emsp;&emsp;这段代码把第 3 章只见过一档的 exec 安全级别补成了完整的"三档 × 三态"。你能看到三种典型搭配各自的定位：`full/off` 是默认的流畅模式，`allowlist/on-miss` 是受控生产的精细模式，`deny/off` 是最严的锁定模式。理解这三档的关键不是记住名字，而是看懂它们背后的**信任梯度**——你越不信任运行环境，就把档位调得越严。下面这张决策树图把"什么场景选哪一档"画出来，帮你建立选型直觉。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102200355.png" width=60%></div>

> **【常见误区】**：以为既然默认是 `full`，那 OpenClaw 就"只能全开"或"改起来很麻烦"。实际上三档是配置项，你可以根据信任边界自由选择——给不信任的非主会话设成 `deny` 或配合 sandbox，给受控生产设成 `allowlist`。后果是如果你以为只能全开，就会在该收紧的场景下不敢用 OpenClaw。正确做法：把三档当成一个信任度旋钮，按环境拧到合适的位置。排查方法：当你担心安全时，先想清楚"这个环境我信任到什么程度"，再对应选档。

> **【Tier 1 验证已通过】**：上面的断言独立验证了 exec 安全配置的三个共性——ExecSecurity 恰好覆盖 `deny`/`allowlist`/`full` 三档、ExecAsk 恰好覆盖 `off`/`on-miss`/`always` 三态、默认值与源码 `exec-approvals.ts` 一致。这是"安全档位枚举完整"的结构性证据。

### 4.2 Docker sandbox：安全体系的另一层纵深

&emsp;&emsp;exec 的三档档位控制的是"agent 能调用哪些命令"，但这只是安全的一层。还有一个更彻底的问题：就算限制了能调哪些命令，这些命令终究是在某个环境里跑的——那这个环境本身能不能被隔离？这就是 OpenClaw 的另一层纵深防御——**Docker sandbox（沙箱）**。它和 exec policy 不是替代关系，而是互补的两层：exec policy 管"能调什么"，sandbox 管"在什么样的容器环境里调"。先用一张最小表把这两个层次压清楚。

&emsp;&emsp;再往源码层看，OpenClaw 的 sandbox 配置在 `src/config/zod-schema.agent-runtime.ts` 里用 zod schema 定义，其中几个字段是安全纵深的关键。`capDrop`（第 175 行）用来删除容器的 Linux capabilities（比如设成 `["ALL"]` 就是删除所有特权，让容器里的进程拿不到危险的内核能力）。`seccompProfile`（第 202 行）指定 seccomp 系统调用过滤规则——值得注意的是，schema 会**主动拒绝**把它设成 `"unconfined"`（第 235 行），也就是不允许你"完全不过滤系统调用"，从源头堵住一个常见的危险配置。`network`（第 173 行，类型是可选字符串 `z.string().optional()`）控制容器的网络模式——schema 在校验时**主动 block 掉 `"host"` 模式**（判断在第 218 行），因为 host 网络模式会让容器直接共享宿主网络栈、隔离形同虚设；它还会默认拦掉 `"container:*"`（复用别的容器网络命名空间，第 226 行）。除这两类被拦下外，其余网络模式——`"none"`（无网络）、`"bridge"`（桥接）乃至自定义的 bridge 网络名——都可接受，并不是只能在 none/bridge 里二选一。还有 `binds`（第 206 行）控制宿主路径挂载，且只允许绝对路径。下面这张表把两层安全的控制对象、关键手段和源码锚点放到一起。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>exec policy 层 vs Docker sandbox 层的职责分工</font></p>
<div class="center">

| 层 | 控制什么 | 关键手段 | 源码锚点 |
|----|----------|----------|----------|
| exec policy | agent 能调用哪些命令 | deny / allowlist / full 三档 + ask 三态 | exec-approvals.ts:24,25 |
| Docker sandbox | 命令在什么样的容器环境里跑 | capDrop 删特权 / seccompProfile 过滤系统调用 / network 限网络 / binds 限挂载 | zod-schema.agent-runtime.ts:175,202,173,206 |

</div>

&emsp;&emsp;这里有一个关于"sandbox 何时生效"的要点要交代清楚，免得你产生误会。

> **【踩坑预警】**：以为 sandbox 是"默认就开、保护所有会话"的。实际上根据 `README.md` 的安全模型说明，默认情况下 `main` 会话的工具是**直接在宿主上跑**的（因为 main 会话就是你自己，全权信任）；sandbox 主要用于**非 main 会话**——你需要显式设置 `agents.defaults.sandbox.mode: "non-main"` 才会让非 main 会话进沙箱（Docker 是默认的 sandbox backend）。后果是如果你误以为"装上就自动全隔离"，可能会在群组/多人渠道场景下放松警惕。正确做法：把 sandbox 理解成"给不完全信任的非主会话准备的隔离层"，按 README 安全文档显式开启。排查方法：在群组或多渠道场景部署前，务必先读 OpenClaw 的 Security 和 Sandboxing 官方文档确认隔离配置。

&emsp;&emsp;关于 sandbox 的具体默认配置组合，`README.md` 的安全模型里给了一个典型示例（`README.md:174` 原文）：非 main 会话的沙箱默认允许 `bash`、`process`、`read`、`write`、`edit`、`sessions_*` 这类会话操作，而拒绝 `browser`、`canvas`、`nodes`、`cron`、`discord`、`gateway` 这类更敏感的能力。这一节我们不在课件里真跑 Docker（学习环境未必有 Docker，且这属于进阶部署场景），只要你理解"sandbox 是叠加在 exec policy 之上的环境隔离层"这个定位就够了。

### 4.3 三对张力收口 + 整节回顾

&emsp;&emsp;补全了安全档位、看清了 sandbox 纵深，我们终于可以把这一节课最高层的东西点透了——OpenClaw 的设计里贯穿着**三对张力**。所谓张力，就是两个都合理、却会互相拉扯的目标，设计者必须在它们之间做取舍。我们深讲其中一对，另外两对只点名字、不展开。

&emsp;&emsp;深讲的这一对是 **信任 vs 约束**——它正是我们这一节课反复触及的主线。回顾一下这条完整的实例链：第 3 章我们看到 exec 默认 `full`/`off`（高信任、低约束）；第 4.1 节我们看到可以收紧到 `allowlist` 甚至 `deny`（降信任、加约束）；第 4.2 节我们看到 Docker sandbox 用 capDrop、seccomp、network 限制做物理层隔离（本课介绍的三层手段中约束最强）。这一整条链揭示了 OpenClaw 的设计哲学：**它的默认值预设"你信任自己的机器、信任自己作为操作者的意图"，所以默认敞开以换取流畅；而约束层是一层层叠加上去的，不是推翻默认，而是让你根据真实的信任边界选择合适的档位。** 对比一下就更清楚——一个多租户的 SaaS 平台必须默认假设用户互相不信任、需要强隔离；而一个个人 trusted-operator 工具默认全开才是合理的，因为操作者就是你自己。这不是疏忽，是针对使用场景做的有意取舍。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260603102208390.png" width=60%></div>

&emsp;&emsp;另外两对张力，我们只点名字，免得现在就把信息塞得太满。一对是 **集中 vs 分治**——Gateway、channel、node 这三层之间如何分摊决策权（谁来集中调度、谁来分散执行），今天先记住这个名字。另一对是 **静态 vs 动态**——plugin 用 manifest 静态声明能力，而 runtime 又需要动态路由，这两者之间存在天然的张力。这两对今天都不展开，你只要知道"OpenClaw 的设计里还有这两条值得追的线"就够了。

> **【常见误区】**：试图现在就把"集中 vs 分治""静态 vs 动态"这两对张力搞透。后果是信息过载——这两对张力涉及的 Gateway 分层、plugin manifest 路由机制都还没铺垫，硬讲只会让你困惑。正确做法：今天只认领它们的名字，不再深究。排查方法：如果你发现自己在追问"那 Gateway 到底怎么分治的"，提醒自己——本节到此为止。

&emsp;&emsp;到这里，这一节课的五章就全部走完了。我们停下来做一次完整的能力回顾——下面这些事，现在的你应该都能做到了。

&emsp;&emsp;关于**定位与认知**（第 0 章）：你能用跨渠道 × 跨设备 × 跨 model provider 三个维度复述 OpenClaw 的定位，能说出它演进四代的名字（Warelay → Clawdbot → Moltbot → OpenClaw）和"蜕壳成长"的隐喻，也能划清它不是 Skill 平台、不是只能连 Slack、device 不是 IoT、ACP 不是自创协议这四条边界。关于**装机实操**（第 1 章）：你能在自己机器上 clone 仓库、用 pnpm 装依赖、配好 provider，最后在仓库内用 `pnpm openclaw tui --local` 发出第一条消息看到工具调用展示（知道全局发布版会滞后于主分支，本课要用源码版跑以保证和讲解完全一致）。关于**运行时心脏**（第 2 章）：你能读懂双层 while 结构、解释 steering 为何是 agent 与 chatbot 的分水岭、叫出 EventStream 的 10 种事件和 6 种主骨架的顺序。关于**工具系统**（第 3 章）：你能分清 plugin / capability / tool 三个概念的边界、说出 8 步策略管线的逻辑顺序、正确框定 exec 默认 `full`/`off` 不是漏洞。关于**安全纵深与哲学**（第 4 章）：你能区分 exec `deny`/`allowlist`/`full` 三档和 `off`/`on-miss`/`always` 三态的适用场景，能讲清 Docker sandbox 作为环境隔离层的定位，也能用"信任 vs 约束"的张力思维解释 OpenClaw 的默认值选择。

&emsp;&emsp;这一节课我们跟着一条消息的生命周期，从认清龙虾的身份，到在你机器上把它跑起来，到剖开它的心脏看 loop 怎么转，到管住它的手看工具怎么受控，最后到理解它为什么这么设计。这条主线走完，本节课就完整收口了——对照上面的能力回顾逐项确认，缺哪块就回去翻对应章节。
